# Check Fields for CDLE


In [1]:
import pandas as pd
from sodapy import Socrata
pd.set_option('display.max_rows', 500) 
pd.set_option('display.max_columns', None)

In [2]:
fin = open("list")
files=fin.readlines()
files

['Ces.csv\n',
 'Income.csv\n',
 'Industry.csv\n',
 'Labforce.csv\n',
 'OESWage.csv\n',
 'lngproj.csv\n',
 'shproj.csv\n']

## Current-Employment-Statistics (CES)

In [3]:
path = "/home/joe/bic_etl/cdle/cdle_LaborAndEmployment/data_transformed/"
file="Ces.csv"
file=file.strip()
ces = pd.read_csv(f"{path}0824/{file}")

In [4]:
ces.head()

,stateabbrv,statename,stfips,areatyname,areaname,area,periodyear,periodtype,pertypdesc,period,seriescode,seriesttls,seriesdesc,adjusted,benchmark,prelim,empces,empprodwrk,empfemale,hours,earnings,hourearn,supprecord,supphe,supppw,suppfem,hoursallwrkr,earningsallwrkr,hourearnallwrkr,suppheallwrkr
0,CO,Colorado,8,State,Colorado,0,1999,3,Monthly,4,30000000,Manufacturing,31-33,0,2023,0,187300,NaN,NaN,NaN,NaN,NaN,0,0,1,1,NaN,NaN,NaN,0
1,CO,Colorado,8,State,Colorado,0,1999,3,Monthly,4,8000000,Private Service Providing,NaN,0,2023,0,1446000,NaN,NaN,NaN,NaN,NaN,0,1,1,1,NaN,NaN,NaN,0
2,CO,Colorado,8,State,Colorado,0,2024,3,Monthly,6,42445000,Food and Beverage Retailers,72,0,2023,1,51700,NaN,NaN,NaN,NaN,NaN,0,1,1,1,NaN,NaN,NaN,1
3,CO,Colorado,8,State,Colorado,0,1999,3,Monthly,4,0,Total Nonfarm,NaN,0,2023,0,2123100,NaN,NaN,NaN,NaN,NaN,0,1,1,1,NaN,NaN,NaN,1
4,CO,Colorado,8,State,Colorado,0,1999,3,Monthly,4,20237000,Heavy and Civil Engineering Construction,237,0,2023,0,19300,NaN,NaN,NaN,NaN,NaN,0,1,1,1,NaN,NaN,NaN,1


In [2]:
def countMiss(df,title):
    dfMiss = df.isna().sum().to_dict()
    dfNrecs = df.shape[0]
    dfNcols = df.shape[1]
    print(f"{title}\nTotal Columns: ",dfNcols)
    for col,cnt in dfMiss.items():
        pp = 100*cnt/dfNrecs
        if cnt > 0:
           print(f"{col:25.25s}  {cnt:8d}  {pp:5.2f}") 

In [16]:
countMiss(ces,"CES")

CES
Total Columns:  30
seriesdesc                    55944  35.96
empprodwrk                   155582  100.00
empfemale                    155582  100.00
hours                        154570  99.35
earnings                     154570  99.35
hourearn                     154570  99.35
hoursallwrkr                 151442  97.34
earningsallwrkr              151442  97.34
hourearnallwrkr              151442  97.34


In [4]:
ces.columns

Index(['stateabbrv', 'statename', 'stfips', 'areatyname', 'areaname', 'area',
       'periodyear', 'periodtype', 'pertypdesc', 'period', 'seriescode',
       'seriesttls', 'seriesdesc', 'adjusted', 'benchmark', 'prelim', 'empces',
       'empprodwrk', 'empfemale', 'hours', 'earnings', 'hourearn',
       'supprecord', 'supphe', 'supppw', 'suppfem', 'hoursallwrkr',
       'earningsallwrkr', 'hourearnallwrkr', 'suppheallwrkr'],
      dtype='object')

In [4]:
ces.shape

(155582, 30)

In [3]:
wfile=f"https://data.colorado.gov/resource/pt2g-89wc.csv?$limit=10000000"
cesCIM = pd.read_csv(wfile)

In [5]:
cesCIM.shape

(154206, 30)

In [15]:
ces["periodyear"].value_counts().sort_index()

periodyear
1990    4431
1991    4431
1992    4431
1993    4431
1994    4431
1995    4431
1996    4431
1997    4431
1998    4431
1999    4431
2000    4457
2001    4457
2002    4470
2003    4535
2004    4561
2005    4561
2006    4561
2007    4561
2008    4561
2009    4561
2010    4561
2011    4561
2012    4561
2013    4561
2014    4561
2015    4561
2016    4561
2017    4561
2018    4561
2019    4535
2020    4535
2021    4535
2022    4535
2023    4431
2024    2367
Name: count, dtype: int64

In [16]:
cesCIM["periodyear"].value_counts().sort_index()


periodyear
1990    4431
1991    4431
1992    4431
1993    4431
1994    4431
1995    4431
1996    4431
1997    4431
1998    4431
1999    4431
2000    4457
2001    4457
2002    4470
2003    4535
2004    4561
2005    4561
2006    4561
2007    4561
2008    4561
2009    4561
2010    4561
2011    4561
2012    4561
2013    4561
2014    4561
2015    4561
2016    4561
2017    4561
2018    4561
2019    4535
2020    4535
2021    4535
2022    4535
2023    4431
2024     991
Name: count, dtype: int64

In [18]:
ces["period"].value_counts()

period
4     12237
6     12237
5     12237
3     12237
2     12237
1     12237
9     11893
10    11893
7     11893
11    11893
12    11893
8     11893
0     10802
Name: count, dtype: int64

In [6]:
cesInvt={}
#years = cesCIM["periodyear"].values.tolist()
#mons = cesCIM["period"].values.tolist()

years = ces["periodyear"].values.tolist()
mons = ces["period"].values.tolist()


for nn,year in enumerate(years):
    mon = mons[nn]
    if year not in cesInvt:
        cesInvt[year]={}
    if mon not in cesInvt[year]:
        cesInvt[year][mon]=0
    cesInvt[year][mon]+=1
        


In [7]:
cesCIMInvt={}
#years = cesCIM["periodyear"].values.tolist()
#mons = cesCIM["period"].values.tolist()

years = cesCIM["periodyear"].values.tolist()
mons = cesCIM["period"].values.tolist()


for nn,year in enumerate(years):
    mon = mons[nn]
    if year not in cesCIMInvt:
        cesCIMInvt[year]={}
    if mon not in cesCIMInvt[year]:
        cesCIMInvt[year][mon]=0
    cesCIMInvt[year][mon]+=1
        

In [8]:
years = list(cesInvt.keys()) + list(cesCIMInvt.keys())
year1 = min(years)
year2 = max(years)
print("YEAR MON  CIM  UPT DIFF")
for year in range(year1,year2+1):
    for mo in range(13):
        if year in cesInvt and mo in cesInvt[year]:
            cnt=cesInvt[year][mo]
        else:
            cnt=0

        if year in cesCIMInvt and mo in cesCIMInvt[year]:
            cntCIM=cesCIMInvt[year][mo]
        else:
            cntCIM=0
        diff = cnt-cntCIM
        print(f"{year:4d}  {mo:2d} {cntCIM:4d} {cnt:4d} {diff:4d}")

YEAR MON  CIM  UPT DIFF
1990   0  303  303    0
1990   1  344  344    0
1990   2  344  344    0
1990   3  344  344    0
1990   4  344  344    0
1990   5  344  344    0
1990   6  344  344    0
1990   7  344  344    0
1990   8  344  344    0
1990   9  344  344    0
1990  10  344  344    0
1990  11  344  344    0
1990  12  344  344    0
1991   0  303  303    0
1991   1  344  344    0
1991   2  344  344    0
1991   3  344  344    0
1991   4  344  344    0
1991   5  344  344    0
1991   6  344  344    0
1991   7  344  344    0
1991   8  344  344    0
1991   9  344  344    0
1991  10  344  344    0
1991  11  344  344    0
1991  12  344  344    0
1992   0  303  303    0
1992   1  344  344    0
1992   2  344  344    0
1992   3  344  344    0
1992   4  344  344    0
1992   5  344  344    0
1992   6  344  344    0
1992   7  344  344    0
1992   8  344  344    0
1992   9  344  344    0
1992  10  344  344    0
1992  11  344  344    0
1992  12  344  344    0
1993   0  303  303    0
1993   1  344  3

In [9]:
year1 = int(min(cesInvt.keys()))
year2 = int(max(cesInvt.keys()))

for year in range(year1,year2+1):
    print(year,end="")
    for mon in range(0,13):
        if year in cesInvt and mon in cesInvt[year]:
            cnt=cesInvt[year][mon]
        else:
            cnt=0
        print(f"{cnt:5d}",end="")
    print("")
        


1990  303  344  344  344  344  344  344  344  344  344  344  344  344
1991  303  344  344  344  344  344  344  344  344  344  344  344  344
1992  303  344  344  344  344  344  344  344  344  344  344  344  344
1993  303  344  344  344  344  344  344  344  344  344  344  344  344
1994  303  344  344  344  344  344  344  344  344  344  344  344  344
1995  303  344  344  344  344  344  344  344  344  344  344  344  344
1996  303  344  344  344  344  344  344  344  344  344  344  344  344
1997  303  344  344  344  344  344  344  344  344  344  344  344  344
1998  303  344  344  344  344  344  344  344  344  344  344  344  344
1999  303  344  344  344  344  344  344  344  344  344  344  344  344
2000  305  346  346  346  346  346  346  346  346  346  346  346  346
2001  305  346  346  346  346  346  346  346  346  346  346  346  346
2002  306  347  347  347  347  347  347  347  347  347  347  347  347
2003  311  352  352  352  352  352  352  352  352  352  352  352  352
2004  313  354  354 

## Income-Data-for-Colorado-Counties (Income)

In [30]:

path = "/home/joe/bic_etl/cdle/cdle_LaborAndEmployment/data_transformed/"
file="Income.csv"
file=file.strip()
income = pd.read_csv(f"{path}0824/{file}")

In [31]:
income.shape

(9022, 19)

In [32]:
income.isna().sum()

stateabbrv        0
statename         0
stfips            0
areatyname        0
areaname          0
areatype          0
area              0
periodyear        0
periodtype        0
pertypdesc        0
period            0
inctype           0
incdesc           0
incsource         0
incsrcdesc        0
income            0
incrank         520
population      520
releasedate    1812
dtype: int64

In [14]:
income.head()

,stateabbrv,statename,stfips,areatyname,areaname,areatype,area,periodyear,periodtype,pertypdesc,period,inctype,incdesc,incsource,incsrcdesc,income,incrank,population,releasedate
0,US,U.S.A.,0,United States,United States,0,0,1929,1,Annual,0,1,Total Personal Income - Bureau of Economic Ana...,3,Bureau of Economic Analysis (BEA),85126000000,0.0,121769000.0,20140612.0
1,US,U.S.A.,0,United States,United States,0,0,1929,1,Annual,0,2,Per Capita Personal Income - Bureau of Economi...,3,Bureau of Economic Analysis (BEA),699,0.0,121769000.0,20140612.0
2,US,U.S.A.,0,United States,United States,0,0,1930,1,Annual,0,1,Total Personal Income - Bureau of Economic Ana...,3,Bureau of Economic Analysis (BEA),76371000000,0.0,123075000.0,20140612.0
3,US,U.S.A.,0,United States,United States,0,0,1930,1,Annual,0,2,Per Capita Personal Income - Bureau of Economi...,3,Bureau of Economic Analysis (BEA),621,0.0,123075000.0,20140612.0
4,US,U.S.A.,0,United States,United States,0,0,1931,1,Annual,0,1,Total Personal Income - Bureau of Economic Ana...,3,Bureau of Economic Analysis (BEA),65507000000,0.0,124038000.0,20140612.0


In [15]:
income["incdesc"].value_counts()

incdesc
Total Personal Income - Bureau of Economic Analysis                   3636
Per Capita Personal Income - Bureau of Economic Analysis              3636
Median Household Income - United States Census                        1718
Dividends, Interest, and Rent Income - Bureau of Economic Analysis      16
Transfer Payments - Bureau of Economic Analysis                         16
Name: count, dtype: int64

In [16]:
income["pertypdesc"].value_counts()

pertypdesc
Annual    9022
Name: count, dtype: int64

In [17]:
years = income["periodyear"].values.tolist()
incomeInvt = {}
for year in years:
    if year not in incomeInvt:
        incomeInvt[year]=0
    incomeInvt[year]+=1

In [18]:
year1=min(incomeInvt.keys())
year2=max(incomeInvt.keys())

for year in range(year1,year2+1):
    if year in incomeInvt:
        cnt=incomeInvt[year]
    else:
        cnt=0
    print(f"{year:4d} {cnt:4d}")

1929    4
1930    4
1931    4
1932    4
1933    4
1934    4
1935    4
1936    4
1937    4
1938    4
1939    4
1940    4
1941    4
1942    4
1943    4
1944    4
1945    4
1946    4
1947    4
1948    4
1949    4
1950    4
1951    4
1952    4
1953    4
1954    4
1955    4
1956    4
1957    4
1958    4
1959    4
1960    4
1961    4
1962    4
1963    4
1964    4
1965    4
1966    4
1967    4
1968    4
1969  146
1970  146
1971  146
1972  146
1973  146
1974  146
1975  146
1976  146
1977  146
1978  146
1979  146
1980  146
1981  146
1982  146
1983  146
1984  148
1985  148
1986  148
1987  148
1988  148
1989  211
1990  211
1991  148
1992  148
1993  211
1994  148
1995  211
1996  148
1997  211
1998  211
1999  211
2000  211
2001  212
2002  212
2003  212
2004  212
2005  212
2006  212
2007  212
2008  216
2009  216
2010  216
2011  215
2012  215
2013  215
2014  215
2015  201
2016  197
2017  132
2018    0
2019    0
2020   65
2021   65


In [19]:
wfile=f"https://data.colorado.gov/resource/2cpa-vbur.csv?$limit=10000000"
incomeCIM = pd.read_csv(wfile)

In [20]:
years = incomeCIM["periodyear"].values.tolist()
incomeCIMInvt = {}
for year in years:
    if year not in incomeCIMInvt:
        incomeCIMInvt[year]=0
    incomeCIMInvt[year]+=1

In [21]:
year1=min(incomeInvt.keys())
year2=max(incomeInvt.keys())

for year in range(year1,year2+1):
    if year in incomeInvt:
        cnt=incomeInvt[year]
    else:
        cnt=0
    print(f"{year:4d} {cnt:4d}")

1929    4
1930    4
1931    4
1932    4
1933    4
1934    4
1935    4
1936    4
1937    4
1938    4
1939    4
1940    4
1941    4
1942    4
1943    4
1944    4
1945    4
1946    4
1947    4
1948    4
1949    4
1950    4
1951    4
1952    4
1953    4
1954    4
1955    4
1956    4
1957    4
1958    4
1959    4
1960    4
1961    4
1962    4
1963    4
1964    4
1965    4
1966    4
1967    4
1968    4
1969  146
1970  146
1971  146
1972  146
1973  146
1974  146
1975  146
1976  146
1977  146
1978  146
1979  146
1980  146
1981  146
1982  146
1983  146
1984  148
1985  148
1986  148
1987  148
1988  148
1989  211
1990  211
1991  148
1992  148
1993  211
1994  148
1995  211
1996  148
1997  211
1998  211
1999  211
2000  211
2001  212
2002  212
2003  212
2004  212
2005  212
2006  212
2007  212
2008  216
2009  216
2010  216
2011  215
2012  215
2013  215
2014  215
2015  201
2016  197
2017  132
2018    0
2019    0
2020   65
2021   65


In [34]:
years = list(incomeCIMInvt.keys()) + list(incomeInvt.keys())
year1=min(years)
year2=max(years)
print("YEAR   CIM   UPT  DIFF")
for year in range(year1,year2+1):
    if year in incomeInvt:
        cnt=incomeInvt[year]
    else:
        cnt=0

    if year in incomeCIMInvt:
        cntCIM = incomeCIMInvt[year]
    else:
        cntCIM=0

    diff=cnt-cntCIM
    print(f"{year:4d} {cntCIM:5d} {cnt:5d} {diff:5d}")
    

YEAR   CIM   UPT  DIFF
1929     4     4     0
1930     4     4     0
1931     4     4     0
1932     4     4     0
1933     4     4     0
1934     4     4     0
1935     4     4     0
1936     4     4     0
1937     4     4     0
1938     4     4     0
1939     4     4     0
1940     4     4     0
1941     4     4     0
1942     4     4     0
1943     4     4     0
1944     4     4     0
1945     4     4     0
1946     4     4     0
1947     4     4     0
1948     4     4     0
1949     4     4     0
1950     4     4     0
1951     4     4     0
1952     4     4     0
1953     4     4     0
1954     4     4     0
1955     4     4     0
1956     4     4     0
1957     4     4     0
1958     4     4     0
1959     4     4     0
1960     4     4     0
1961     4     4     0
1962     4     4     0
1963     4     4     0
1964     4     4     0
1965     4     4     0
1966     4     4     0
1967     4     4     0
1968     4     4     0
1969   146   146     0
1970   146   146     0
1971   146 

## Occupational-Employment-Statistics (OESWage)

### Read Local

In [17]:
path = "/home/joe/bic_etl/cdle/cdle_LaborAndEmployment/data_transformed/"
file="OESWage.csv"
file=file.strip()
oes = pd.read_csv(f"{path}0824/{file}",low_memory=False)

In [18]:
oes.shape

(329410, 40)

In [19]:
oes.isna().sum()

stateabbrv           0
statename            0
stfips               0
areaname             0
areatype             0
areatyname           0
area                 0
periodyear           0
periodtype           0
pertypdesc           0
period               0
indcodty             0
indcode              0
indcodetitle         0
occodetype           0
occodetydesc         0
occcode              0
codetitle            0
wagesource           0
wagesrdesc           0
ratetype             0
ratetydesc           0
empcount           545
response            70
mean              8248
entrywg           8739
experience        9212
pct10             8338
pct25             8760
median            9701
pct75            11169
pct90            16084
udpct            47374
udpctwage       320890
udrnglopct       47374
udrnghipct       47374
udrngmean       320890
wpctrelerr        1674
epctrelerr         825
panelcode        26846
dtype: int64

In [52]:
oes.head()

,stateabbrv,statename,stfips,areaname,areatype,areatyname,area,periodyear,periodtype,pertypdesc,period,indcodty,indcode,indcodetitle,occodetype,occodetydesc,occcode,codetitle,wagesource,wagesrdesc,ratetype,ratetydesc,empcount,response,mean,entrywg,experience,pct10,pct25,median,pct75,pct90,udpct,udpctwage,udrnglopct,udrnghipct,udrngmean,wpctrelerr,epctrelerr,panelcode
0,CO,Colorado,8,Colorado Springs MSA,21,Metropolitan Statistical Area,17820,2011,1,Annual,0,10,0,Total All,8,SOC 2000,0,Total All occupations,3,Occupational Employment Statistics Survey,4,Annual wage or salary,241620.0,77.0,45082.00,20375.00,57435.00,17996.00,22990.00,35100.00,57044.00,86582.00,NaN,NaN,NaN,NaN,NaN,2.03,1.09,NaN
1,CO,Colorado,8,Colorado Springs MSA,21,Metropolitan Statistical Area,17820,2011,1,Annual,0,10,0,Total All,8,SOC 2000,0,Total All occupations,3,Occupational Employment Statistics Survey,1,Hourly wage,241620.0,77.0,21.67,9.80,27.61,8.65,11.05,16.88,27.42,41.63,NaN,NaN,NaN,NaN,NaN,2.03,1.09,NaN
2,CO,Colorado,8,Pueblo MSA,21,Metropolitan Statistical Area,39380,2009,1,Annual,0,10,0,Total All,8,SOC 2000,0,Total All occupations,3,Occupational Employment Statistics Survey,4,Annual wage or salary,56230.0,79.0,36531.00,18354.00,45620.00,16427.00,20052.00,30308.00,45354.00,64700.00,0.0,NaN,0.0,0.0,NaN,2.12,1.95,200905.0
3,CO,Colorado,8,Fort Collins-Loveland MSA,21,Metropolitan Statistical Area,22660,2011,1,Annual,0,10,0,Total All,8,SOC 2000,0,Total All occupations,3,Occupational Employment Statistics Survey,1,Hourly wage,127420.0,77.0,21.39,9.64,27.27,8.58,10.85,16.77,26.40,39.76,NaN,NaN,NaN,NaN,NaN,2.23,1.27,NaN
4,CO,Colorado,8,Fort Collins-Loveland MSA,21,Metropolitan Statistical Area,22660,2011,1,Annual,0,10,0,Total All,8,SOC 2000,0,Total All occupations,3,Occupational Employment Statistics Survey,4,Annual wage or salary,127420.0,77.0,44499.00,20049.00,56724.00,17844.00,22571.00,34891.00,54906.00,82692.00,NaN,NaN,NaN,NaN,NaN,2.23,1.27,NaN


In [53]:
oes["period"].value_counts()

period
0    329410
Name: count, dtype: int64

In [67]:
oesInvt=oes["periodyear"].value_counts().sort_index().to_dict()


In [57]:
oes["areatyname"].value_counts()

areatyname
County                           104639
Metropolitan Statistical Area     88591
Workforce Development Region      80295
State                             21898
Balance of State                  18118
Balance of State (pre-2015)       15869
Name: count, dtype: int64

In [58]:
oes.groupby(["periodyear","areatyname"])["period"].count()

periodyear  areatyname                   
2009        Balance of State (pre-2015)       1732
            Metropolitan Statistical Area     5435
            State                             1377
2010        Balance of State (pre-2015)       2430
            Metropolitan Statistical Area     5436
            State                             1418
2011        Balance of State (pre-2015)       2225
            Metropolitan Statistical Area     5308
            State                             1361
2012        Balance of State (pre-2015)       2472
            Metropolitan Statistical Area     5714
            State                             1494
2013        Balance of State (pre-2015)       2386
            Metropolitan Statistical Area     5770
            State                             1490
2014        Balance of State (pre-2015)       2476
            Metropolitan Statistical Area     5756
            State                             1490
2015        Balance of State            

### Read CIM

In [5]:
wfile=f"https://data.colorado.gov/resource/busm-qa5b.csv?$limit=10000000"
oesCIM = pd.read_csv(wfile,low_memory=False)

In [68]:
oesCIMInvt=oesCIM["periodyear"].value_counts().sort_index().to_dict()

In [79]:
years =  list(oesCIMInvt.keys()) + list(oesInvt.keys())
year1=min(years)
year2=max(years)
print("YEAR  CIM   UPDT   DIFF")
for year in range(year1,year2+1):
    if year in oesCIMInvt:
        cntCIM=oesCIMInvt[year]
    else:
        cntCIM=0
    if year in oesInvt:
        cnt=oesInvt[year]
    else:
        cnt=0
    diff = cnt-cntCIM
    print(f"{year:4d} {cntCIM:5d} {cnt:5d}  {diff:5d}")

YEAR  CIM   UPDT   DIFF
2009  8544  8544      0
2010  9284  9284      0
2011  8894  8894      0
2012  9680  9680      0
2013  9646  9646      0
2014  9722  9722      0
2015  9432  9432      0
2016  9358  9358      0
2017  9350  9350      0
2018 36858 37036    178
2019 37618 37838    220
2020 31070 31287    217
2021 47217 47217      0
2022 47388 47388      0
2023     0 44734  44734


In [106]:
oes.head()

,stateabbrv,statename,stfips,areaname,areatype,areatyname,area,periodyear,periodtype,pertypdesc,period,indcodty,indcode,indcodetitle,occodetype,occodetydesc,occcode,codetitle,wagesource,wagesrdesc,ratetype,ratetydesc,empcount,response,mean,entrywg,experience,pct10,pct25,median,pct75,pct90,udpct,udpctwage,udrnglopct,udrnghipct,udrngmean,wpctrelerr,epctrelerr,panelcode
0,CO,Colorado,8,Colorado Springs MSA,21,Metropolitan Statistical Area,17820,2011,1,Annual,0,10,0,Total All,8,SOC 2000,000000,Total All occupations,3,Occupational Employment Statistics Survey,4,Annual wage or salary,241620.0,77.0,45082.00,20375.00,57435.00,17996.00,22990.00,35100.00,57044.00,86582.00,NaN,NaN,NaN,NaN,NaN,2.03,1.09,NaN
1,CO,Colorado,8,Colorado Springs MSA,21,Metropolitan Statistical Area,17820,2011,1,Annual,0,10,0,Total All,8,SOC 2000,000000,Total All occupations,3,Occupational Employment Statistics Survey,1,Hourly wage,241620.0,77.0,21.67,9.80,27.61,8.65,11.05,16.88,27.42,41.63,NaN,NaN,NaN,NaN,NaN,2.03,1.09,NaN
2,CO,Colorado,8,Pueblo MSA,21,Metropolitan Statistical Area,39380,2009,1,Annual,0,10,0,Total All,8,SOC 2000,000000,Total All occupations,3,Occupational Employment Statistics Survey,4,Annual wage or salary,56230.0,79.0,36531.00,18354.00,45620.00,16427.00,20052.00,30308.00,45354.00,64700.00,0.0,NaN,0.0,0.0,NaN,2.12,1.95,200905.0
3,CO,Colorado,8,Fort Collins-Loveland MSA,21,Metropolitan Statistical Area,22660,2011,1,Annual,0,10,0,Total All,8,SOC 2000,000000,Total All occupations,3,Occupational Employment Statistics Survey,1,Hourly wage,127420.0,77.0,21.39,9.64,27.27,8.58,10.85,16.77,26.40,39.76,NaN,NaN,NaN,NaN,NaN,2.23,1.27,NaN
4,CO,Colorado,8,Fort Collins-Loveland MSA,21,Metropolitan Statistical Area,22660,2011,1,Annual,0,10,0,Total All,8,SOC 2000,000000,Total All occupations,3,Occupational Employment Statistics Survey,4,Annual wage or salary,127420.0,77.0,44499.00,20049.00,56724.00,17844.00,22571.00,34891.00,54906.00,82692.00,NaN,NaN,NaN,NaN,NaN,2.23,1.27,NaN


In [114]:
oes["areatype"].value_counts()

areatype
4     104639
21     88591
17     80295
1      21898
75     18118
30     15869
Name: count, dtype: int64

In [24]:
a=oes.loc[oes["periodyear"] == 2018]
b=oesCIM.loc[oesCIM["periodyear"] == 2018]


In [ ]:
a["area"].value_counts()

In [5]:
a.groupby(["area","occcode","indcode","ratetydesc"])["statename"].count()

area   occcode  indcode  ratetydesc           
0      000000   0        Annual wage or salary    1
                         Hourly wage              1
       110000   0        Annual wage or salary    1
                         Hourly wage              1
       111011   0        Annual wage or salary    1
                                                 ..
80003  537064   0        Hourly wage              1
       537072   0        Annual wage or salary    1
                         Hourly wage              1
       537081   0        Annual wage or salary    1
                         Hourly wage              1
Name: statename, Length: 37036, dtype: int64

In [25]:
hit=0
c=a
d=b
indexes=[]
for index,row in c.iterrows():
    area=row["area"]
   
    ocode=row["occcode"].strip()
    rate=row["ratetydesc"].strip()
    tmp=b.loc[(b["area"]==area) & (b["occcode"]==ocode) & (b["ratetydesc"] == rate)]
  #  tmp=d.loc[(d["area"]==area) &  (d["ownership"]==owner)]
    
    
    if tmp.shape[0] == 0:
#        print(index,row)
        hit+=1
        indexes.append(index)
        if hit < 10:
            print(area,ocode,rate)
print("HIT ",hit)

95 119131 Annual wage or salary
95 119131 Hourly wage
11 119131 Annual wage or salary
11 119131 Hourly wage
9 119131 Annual wage or salary
73 119131 Hourly wage
73 119131 Annual wage or salary
121 119131 Hourly wage
121 119131 Annual wage or salary
HIT  178


In [26]:
len(indexes)

178

In [27]:
tmp = oes.loc[oes.index.isin(indexes)]

In [28]:
tmp.shape

(178, 40)

In [29]:
tmp.to_csv("oes-extra.csv",index=False)

In [30]:
tmp.head()

,stateabbrv,statename,stfips,areaname,areatype,areatyname,area,periodyear,periodtype,pertypdesc,period,indcodty,indcode,indcodetitle,occodetype,occodetydesc,occcode,codetitle,wagesource,wagesrdesc,ratetype,ratetydesc,empcount,response,mean,entrywg,experience,pct10,pct25,median,pct75,pct90,udpct,udpctwage,udrnglopct,udrnghipct,udrngmean,wpctrelerr,epctrelerr,panelcode
27120,CO,Colorado,8,Phillips County,4,County,95,2018,1,Annual,0,10,0,Total All,14,SOC 2010,119131,Postmasters and Mail Superintendents,3,Occupational Employment Statistics Survey,4,Annual wage or salary,NaN,100.0,72010.00,64230.00,75899.00,64235.00,64241.00,72020.00,79799.00,79805.00,0.0,NaN,0.0,0.0,NaN,NaN,NaN,201805.0
27131,CO,Colorado,8,Phillips County,4,County,95,2018,1,Annual,0,10,0,Total All,14,SOC 2010,119131,Postmasters and Mail Superintendents,3,Occupational Employment Statistics Survey,1,Hourly wage,NaN,100.0,34.62,30.88,36.49,30.88,30.88,34.62,38.37,38.37,0.0,NaN,0.0,0.0,NaN,NaN,NaN,201805.0
27143,CO,Colorado,8,Bent County,4,County,11,2018,1,Annual,0,10,0,Total All,14,SOC 2010,119131,Postmasters and Mail Superintendents,3,Occupational Employment Statistics Survey,4,Annual wage or salary,NaN,100.0,76149.00,76149.00,76149.00,76151.00,76154.00,76159.00,76164.00,76168.00,0.0,NaN,0.0,0.0,NaN,NaN,NaN,201805.0
27144,CO,Colorado,8,Bent County,4,County,11,2018,1,Annual,0,10,0,Total All,14,SOC 2010,119131,Postmasters and Mail Superintendents,3,Occupational Employment Statistics Survey,1,Hourly wage,NaN,100.0,36.61,36.61,36.61,36.61,36.61,36.62,36.62,36.62,0.0,NaN,0.0,0.0,NaN,NaN,NaN,201805.0
27145,CO,Colorado,8,Baca County,4,County,9,2018,1,Annual,0,10,0,Total All,14,SOC 2010,119131,Postmasters and Mail Superintendents,3,Occupational Employment Statistics Survey,4,Annual wage or salary,NaN,100.0,64168.00,62546.00,64979.00,62550.00,62556.00,64178.00,65801.00,65807.00,0.0,NaN,0.0,0.0,NaN,NaN,NaN,201805.0


In [31]:
oes["panelcode"].value_counts()

panelcode
202205.0    47388
202105.0    47217
202305.0    44734
201905.0    37838
201805.0    37036
202005.0    31287
201405.0     9722
201205.0     9680
201305.0     9646
201605.0     9358
201705.0     9350
201005.0     9284
200905.0       24
Name: count, dtype: int64

## Employment-by-Industry-from-Census-of-Employment (Industry)

### Read Local FIle

In [22]:
path = "/home/joe/bic_etl/cdle/cdle_LaborAndEmployment/data_transformed/"
file="Industry.csv"
file=file.strip()
industry = pd.read_csv(f"{path}0824/{file}",low_memory=False)              

In [11]:
industry["ownership"].value_counts()

ownership
50    577305
00    535237
10    116773
30     97153
20     47464
G0     14190
80      6708
Name: count, dtype: int64

In [12]:
industry.shape

(1394830, 29)

In [13]:
industry.head()

,stateabbrv,statename,stfips,areaname,areatype,areatyname,area,periodyear,periodtype,period,pertypdesc,indcodty,indcode,codetitle,ownership,ownertitle,prelim,firms,estab,avgemp,mnth1emp,mnth2emp,mnth3emp,topempav,totwage,avgwkwage,taxwage,contrib,suppress
0,CO,Colorado,8,Boulder County ...,4,County,13,2019,2,3,Quarterly,10,624,Social Assistance ...,00,Aggregate of all types,0,0,326,4377.0,4464,4474,4193,0.0,30167004.0,530.0,0.0,0.0,0
1,CO,Colorado,8,Boulder County ...,4,County,13,2019,2,3,Quarterly,10,624,Social Assistance ...,50,Private,0,0,326,4377.0,4464,4474,4193,0.0,30167004.0,530.0,0.0,0.0,0
2,CO,Colorado,8,Boulder County ...,4,County,13,2019,2,3,Quarterly,10,71,"Arts, Entertainment, and Recreation ...",00,Aggregate of all types,0,0,340,3424.0,3597,3473,3201,0.0,25780758.0,579.0,0.0,0.0,0
3,CO,Colorado,8,Boulder County ...,4,County,13,2019,2,3,Quarterly,10,71,"Arts, Entertainment, and Recreation ...",10,Federal Government,0,0,1,3.0,3,3,3,0.0,91612.0,2349.0,0.0,0.0,0
4,CO,Colorado,8,Boulder County ...,4,County,13,2019,2,3,Quarterly,10,71,"Arts, Entertainment, and Recreation ...",50,Private,0,0,338,3409.0,3584,3458,3184,0.0,25574113.0,577.0,0.0,0.0,0


In [14]:
industry["pertypdesc"].value_counts()

pertypdesc
Quarterly    1125286
Annual        269544
Name: count, dtype: int64

In [15]:
industry["period"].value_counts()

period
3    287032
2    286842
1    283038
0    269544
4    268374
Name: count, dtype: int64

### Read CIM

In [16]:
wfile=f"https://data.colorado.gov/resource/cjkq-q9ih.csv?$limit=10000000"
industryCIM = pd.read_csv(wfile,low_memory=False)

In [17]:
industryCIM["ownership"].value_counts()

ownership
50    558315
00    514714
10    113337
30     95177
20     46614
G0     14190
80      4001
Name: count, dtype: int64

In [18]:
industryCIM.shape

(1346348, 29)

In [19]:
print(industry.shape[0]-industryCIM.shape[0])

48482


In [28]:
industry.isna().sum()

stateabbrv       0
statename        0
stfips           0
areaname         0
areatype         0
areatyname       0
area             0
periodyear       0
periodtype       0
period           0
pertypdesc       0
indcodty         0
indcode          0
codetitle        0
ownership        0
ownertitle       0
prelim           0
firms            0
estab            0
avgemp          11
mnth1emp         0
mnth2emp         0
mnth3emp         0
topempav      4009
totwage         11
avgwkwage      292
taxwage         11
contrib         11
suppress         0
dtype: int64

In [49]:
industry.loc[industry["avgwkwage"].isnull(),"avgwkwage"]

1588      NaN
1589      NaN
1637      NaN
1638      NaN
2062      NaN
2611      NaN
29655     NaN
29656     NaN
29817     NaN
29818     NaN
29896     NaN
29907     NaN
29908     NaN
93651     NaN
93652     NaN
94307     NaN
94308     NaN
100982    NaN
100985    NaN
100986    NaN
101034    NaN
101037    NaN
101038    NaN
101106    NaN
101109    NaN
101110    NaN
101182    NaN
101185    NaN
101186    NaN
128066    NaN
128119    NaN
128122    NaN
128123    NaN
182689    NaN
182690    NaN
183102    NaN
183103    NaN
183467    NaN
183468    NaN
185639    NaN
185640    NaN
185821    NaN
185822    NaN
186306    NaN
186307    NaN
186510    NaN
186511    NaN
186716    NaN
186717    NaN
229069    NaN
229070    NaN
229141    NaN
229142    NaN
229227    NaN
229228    NaN
229312    NaN
229313    NaN
286061    NaN
286062    NaN
286212    NaN
286213    NaN
286572    NaN
286573    NaN
295938    NaN
295939    NaN
296366    NaN
296367    NaN
308267    NaN
308268    NaN
308383    NaN
308384    NaN
308706

In [24]:
w=industry["avgwkwage"].value_counts().to_dict()

In [ ]:
for k,v in sorted(w.items()):
    print(k)

In [9]:
a=industry.groupby(["periodyear","period"])["area"].count().to_dict()
industryInvt={}
for key,val in a.items():
    yr=key[0]
    mo=key[1]
    if yr not in industryInvt:
        industryInvt[yr]={}
    industryInvt[yr][mo]=val
    

In [10]:
a=industryCIM.groupby(["periodyear","period"])["area"].count().to_dict()
industryCIMInvt={}
for key,val in a.items():
    yr=key[0]
    mo=key[1]
    if yr not in industryCIMInvt:
        industryCIMInvt[yr]={}
    industryCIMInvt[yr][mo]=val

In [11]:
years = list(industryCIMInvt.keys()) + list(industryInvt.keys())
year1 = min(years)
year2 = max(years)
print("YEAR QTR   CIM  UPDT  DIFF")
for year in range(year1,year2+1):
    for qtr in range(5):
        if year in industryInvt and qtr in industryInvt[year]:
            cnt=industryInvt[year][qtr]
        else:
            cnt=0

        if year in industryCIMInvt and qtr in industryCIMInvt[year]:
            cntCIM=industryCIMInvt[year][qtr]
        else:
            cntCIM=0
        diff = cnt-cntCIM
        print(f"{year:4d}   {qtr:1d} {cntCIM:5d} {cnt:5d} {diff:5d}")

YEAR QTR   CIM  UPDT  DIFF
2000   0 12531 12531     0
2000   1 10166 10166     0
2000   2 12684 12684     0
2000   3 12705 12705     0
2000   4 12608 12608     0
2001   0 12589 12589     0
2001   1 12666 12666     0
2001   2 12732 12732     0
2001   3 12699 12699     0
2001   4 12691 12691     0
2002   0 12820 12820     0
2002   1 12826 12826     0
2002   2 12912 12912     0
2002   3 12945 12945     0
2002   4 12907 12907     0
2003   0 12874 12874     0
2003   1 12930 12930     0
2003   2 13026 13026     0
2003   3 12973 12973     0
2003   4 12960 12960     0
2004   0 12884 12884     0
2004   1 12989 12989     0
2004   2 12984 12984     0
2004   3 13004 13004     0
2004   4 12954 12954     0
2005   0 12856 12856     0
2005   1 12928 12928     0
2005   2 12947 12947     0
2005   3 12931 12931     0
2005   4 10183 10183     0
2006   0 12841 12841     0
2006   1 12933 12933     0
2006   2 12991 12991     0
2006   3 12982 12982     0
2006   4 12948 12948     0
2007   0 12943 12943     0
2

In [85]:
industry["ownership"].value_counts()

ownership
50    577305
00    535237
10    116773
30     97153
20     47464
G0     14190
80      6708
Name: count, dtype: int64

In [77]:
for col in ["ownership","indcode"]:
    industryCIM[col]=industryCIM[col].astype(str)
    industry[col]=industry[col].astype(str)


In [15]:
a=industry.loc[(industry["periodyear"] == 2022) & (industry["period"] == 4)]
b=industryCIM.loc[(industryCIM["periodyear"] == 2022) & (industryCIM["period"] == 4)]


In [ ]:
print(a["indcode"].value_counts())
print(b["indcode"].value_counts())


In [ ]:

hit=0
for index,row in industry.iterrows():
    area=row["area"]
    code=row["indcode"]
    owner=row["ownership"]
    tmp=industryCIM.loc[(industryCIM["area"]==area) & (industryCIM["indcode"]==code) & (industryCIM["ownership"]==owner)]
    if tmp.shape[0] == 0:
        print(index,row)
        hit+=1
print("HIT ",hit)

In [16]:
hit=0
c=a
d=b
indexes=[]
for index,row in c.iterrows():
    area=row["area"]
    code=row["indcode"].strip()
    owner=row["ownership"].strip()
    tmp=b.loc[(b["area"]==area) & (b["indcode"].str.strip() ==code) & (b["ownership"]==owner)]
  #  tmp=d.loc[(d["area"]==area) &  (d["ownership"]==owner)]
    
    
    if tmp.shape[0] == 0:
#        print(index,row)
        hit+=1
        indexes.append(index)
        if hit < 10:
            print(area,owner,code)
print("HIT ",hit)

25 10 1028
3 00 1029
3 50 1029
67 00 1029
67 50 1029
53 10 1028
79 10 1028
91 00 1029
91 50 1029
HIT  16


In [100]:
indexes

[128066,
 186716,
 186717,
 672527,
 672528,
 848320,
 909031,
 1064708,
 1064709,
 1175541,
 1175542,
 1269784,
 1269785,
 1346176,
 1374380,
 1374381]

In [17]:
tmp=industry.loc[industry.index.isin(indexes)]

In [18]:
tmp.shape

(16, 29)

In [19]:
tmp.to_csv("industry-extra.csv",index=False)

In [21]:
tmp.head()

,stateabbrv,statename,stfips,areaname,areatype,areatyname,area,periodyear,periodtype,period,pertypdesc,indcodty,indcode,codetitle,ownership,ownertitle,prelim,firms,estab,avgemp,mnth1emp,mnth2emp,mnth3emp,topempav,totwage,avgwkwage,taxwage,contrib,suppress
128066,CO,Colorado,8,Crowley County ...,4,County,25,2022,2,4,Quarterly,10,1028,Public Administration ...,10,Federal Government,0,1,1,0.0,0,0,0,0.0,0.0,NaN,0.0,0.0,0
186716,CO,Colorado,8,Alamosa County ...,4,County,3,2022,2,4,Quarterly,10,1029,Unclassified ...,00,Aggregate of all types,0,1,1,0.0,0,0,0,0.0,0.0,NaN,0.0,0.0,0
186717,CO,Colorado,8,Alamosa County ...,4,County,3,2022,2,4,Quarterly,10,1029,Unclassified ...,50,Private,0,1,1,0.0,0,0,0,0.0,0.0,NaN,0.0,0.0,0
672527,CO,Colorado,8,La Plata County ...,4,County,67,2022,2,4,Quarterly,10,1029,Unclassified ...,00,Aggregate of all types,0,1,1,0.0,0,0,0,0.0,0.0,NaN,0.0,0.0,0
672528,CO,Colorado,8,La Plata County ...,4,County,67,2022,2,4,Quarterly,10,1029,Unclassified ...,50,Private,0,1,1,0.0,0,0,0,0.0,0.0,NaN,0.0,0.0,0


KeyError: 'panelcode'

## Employment-and-Unemployment-Estimates (Labforce)

In [24]:
path = "/home/joe/bic_etl/cdle/cdle_LaborAndEmployment/data_transformed/"
file="Labforce.csv"
file=file.strip()
labf = pd.read_csv(f"{path}0824/{file}")

In [38]:
labf.shape

(35918, 18)

In [39]:
labf.isna().sum()

stateabbrv    0
statename     0
stfips        0
areaname      0
areatype      0
areatyname    0
area          0
periodyear    0
periodtype    0
pertypdesc    0
period        0
adjusted      0
prelim        0
benchmark     0
laborforce    0
emplab        0
unemp         0
unemprate     0
dtype: int64

In [26]:
labf.head()

,stateabbrv,statename,stfips,areaname,areatype,areatyname,area,periodyear,periodtype,pertypdesc,period,adjusted,prelim,benchmark,laborforce,emplab,unemp,unemprate
0,CO,Colorado,8,Colorado,1,State,0,1976,1,Annual,0,0,0,2023,1240720,1166629,74091,6.0
1,CO,Colorado,8,Colorado,1,State,0,1976,3,Monthly,1,0,0,2023,1212048,1126230,85818,7.1
2,CO,Colorado,8,Colorado,1,State,0,1976,3,Monthly,1,1,0,2023,1230966,1160104,70862,5.8
3,CO,Colorado,8,Colorado,1,State,0,1976,3,Monthly,2,0,0,2023,1210264,1130641,79623,6.6
4,CO,Colorado,8,Colorado,1,State,0,1976,3,Monthly,2,1,0,2023,1230680,1160001,70679,5.7


In [27]:
labf["period"].value_counts()

period
1     2828
3     2828
2     2828
4     2828
5     2828
6     2828
9     2748
7     2748
8     2748
11    2748
10    2748
12    2748
0     2462
Name: count, dtype: int64

In [28]:
a=labf.groupby(["periodyear","period"])["area"].count().to_dict()
labfInvt={}
for key,val in a.items():
    yr=key[0]
    mo=key[1]
    if yr not in labfInvt:
       labfInvt[yr]={}
    labfInvt[yr][mo]=val

In [29]:
wfile=f"https://data.colorado.gov/resource/4e3w-qire.csv?$limit=10000000"
labfCIM = pd.read_csv(wfile)

In [30]:
a=labfCIM.groupby(["periodyear","period"])["area"].count().to_dict()
labfCIMInvt={}
for key,val in a.items():
    yr=key[0]
    mo=key[1]
    if yr not in labfCIMInvt:
       labfCIMInvt[yr]={}
    labfCIMInvt[yr][mo]=val

In [31]:
years = list(labfCIMInvt.keys()) + list(labfInvt.keys())
year1 = min(years)
year2 = max(years)
print("YEAR MON   CIM  UPDT  DIFF")
for year in range(year1,year2+1):
    for qtr in range(13):
        if year in labfInvt and qtr in labfInvt[year]:
            cnt=labfInvt[year][qtr]
        else:
            cnt=0

        if year in labfCIMInvt and qtr in labfCIMInvt[year]:
            cntCIM=labfCIMInvt[year][qtr]
        else:
            cntCIM=0
        diff = cnt-cntCIM
        print(f"{year:4d}   {qtr:2d} {cntCIM:5d} {cnt:5d} {diff:5d}")

YEAR MON   CIM  UPDT  DIFF
1976    0     1     1     0
1976    1     2     2     0
1976    2     2     2     0
1976    3     2     2     0
1976    4     2     2     0
1976    5     2     2     0
1976    6     2     2     0
1976    7     2     2     0
1976    8     2     2     0
1976    9     2     2     0
1976   10     2     2     0
1976   11     2     2     0
1976   12     2     2     0
1977    0     1     1     0
1977    1     2     2     0
1977    2     2     2     0
1977    3     2     2     0
1977    4     2     2     0
1977    5     2     2     0
1977    6     2     2     0
1977    7     2     2     0
1977    8     2     2     0
1977    9     2     2     0
1977   10     2     2     0
1977   11     2     2     0
1977   12     2     2     0
1978    0     1     1     0
1978    1     2     2     0
1978    2     2     2     0
1978    3     2     2     0
1978    4     2     2     0
1978    5     2     2     0
1978    6     2     2     0
1978    7     2     2     0
1978    8     2     2

## Long-Term Employment Projections in Colorado (lngproj)

In [2]:
path = "/home/joe/bic_etl/cdle/cdle_LaborAndEmployment/data_transformed/"
path=""
file="lngproj.csv"
file=file.strip()
#long = pd.read_csv(f"{path}0824/{file}")
long = pd.read_csv(f"{file}")


In [3]:
long.shape

(91372, 29)

In [5]:
long.isna().sum()

stateabbrv        0
statename         0
stfips            0
areatype          0
areatyname        0
area              0
areaname          0
periodid          0
perioddesc        0
periodtype        0
matincodty        0
matincode         0
matintitle        0
matoccodty        0
matoccode         0
matocctitl        0
estemp            0
projemp           0
pctestind         0
pctestocc         0
pctprojind        0
pctprojocc        0
nchg              0
pchg              0
growrate          0
aopeng        91372
aopenr        91372
aopent        91372
suppress          0
dtype: int64

In [4]:
long.head()

,stateabbrv,statename,stfips,areatype,areatyname,area,areaname,periodid,perioddesc,periodtype,matincodty,matincode,matintitle,matoccodty,matoccode,matocctitl,estemp,projemp,pctestind,pctestocc,pctprojind,pctprojocc,nchg,pchg,growrate,aopeng,aopenr,aopent,suppress
0,CO,Colorado,8,21,Metropolitan Statistical Area,17820,Colorado Springs MSA,32,"Colorado Long-Term Projections, 2022-2032, Jul...",1,2,900000,Government,2,0,"Total, All Occupations",25229,26695,100.0,7.00,100.0,6.58,1466,5.8108,0.5664,NaN,NaN,NaN,0
1,CO,Colorado,8,21,Metropolitan Statistical Area,17820,Colorado Springs MSA,32,"Colorado Long-Term Projections, 2022-2032, Jul...",1,2,910000,Total Federal Government Employment,2,0,"Total, All Occupations",12343,12616,100.0,3.42,100.0,3.11,273,2.2118,0.2190,NaN,NaN,NaN,0
2,CO,Colorado,8,21,Metropolitan Statistical Area,17820,Colorado Springs MSA,32,"Colorado Long-Term Projections, 2022-2032, Jul...",1,2,930000,"Local Government, Excluding Education and Hosp...",2,0,"Total, All Occupations",10772,11819,100.0,2.99,100.0,2.91,1047,9.7196,0.9319,NaN,NaN,NaN,0
3,CO,Colorado,8,21,Metropolitan Statistical Area,17820,Colorado Springs MSA,32,"Colorado Long-Term Projections, 2022-2032, Jul...",1,2,999300,"Local Government, Excluding Education and Hosp...",2,0,"Total, All Occupations",10772,11819,100.0,2.99,100.0,2.91,1047,9.7196,0.9319,NaN,NaN,NaN,0
4,CO,Colorado,8,21,Metropolitan Statistical Area,17820,Colorado Springs MSA,32,"Colorado Long-Term Projections, 2022-2032, Jul...",1,2,920000,"State Government, Excluding Education and Hosp...",2,0,"Total, All Occupations",2114,2260,100.0,0.59,100.0,0.56,146,6.9063,0.6701,NaN,NaN,NaN,0


In [6]:
long["perioddesc"].value_counts()

perioddesc
Colorado Long-Term Projections, 2023-2033, July 2024    91372
Name: count, dtype: int64

In [12]:
longInvt=long["areaname"].value_counts().to_dict()

In [4]:
wfile=f"https://data.colorado.gov/resource/gyeb-jc69.csv?$limit=10000000"
longCIM = pd.read_csv(wfile)

In [5]:
longCIM.shape

(96797, 29)

In [10]:
longCIMInvt=longCIM["areaname"].value_counts().to_dict()


In [13]:
for area,cnt in longInvt.items():
    if area in longCIMInvt:
        cntCIM = longCIMInvt[area]
    else:
        cntCIM=0
    diff=cnt-cntCIM
    print(f"{area:30s} {cntCIM:5d}  {cnt:5d}  {diff:5d}")

Colorado                       20541  17623  -2918
Pueblo MSA                      4532  15698  11166
Denver - Aurora MSA            16016  13652  -2364
Colorado Springs MSA            9278   6871  -2407
Boulder-Longmont MSA            8257   5982  -2275
Fort Collins-Loveland MSA       7856   5891  -1965
Northwest Colorado              6592   5817   -775
Southwest Colorado              6537   5518  -1019
Greeley MSA                     6220   5161  -1059
Eastern and Southern Colorado   5686   4875   -811
Grand Junction MSA              5282   4284   -998


In [16]:
longCIM.columns

Index(['stateabbrv', 'statename', 'stfips', 'areatype', 'areatyname', 'area',
       'areaname', 'periodid', 'perioddesc', 'periodtype', 'matincodty',
       'matincode', 'matintitle', 'matoccodty', 'matoccode', 'matocctitl',
       'estemp', 'projemp', 'pctestind', 'pctprojind', 'nchg', 'aopeng',
       'aopenr', 'aopent', 'suppress', 'pctprojocc', 'pctestocc', 'growrate',
       'pchg'],
      dtype='object')

In [23]:
icim = longCIM["matintitle"].value_counts().to_dict()

In [24]:
inew = long["matintitle"].value_counts().to_dict()

In [29]:
long.loc[long["matintitle"] == "Total All Industries"]

,stateabbrv,statename,stfips,areatype,areatyname,area,areaname,periodid,perioddesc,periodtype,matincodty,matincode,matintitle,matoccodty,matoccode,matocctitl,estemp,projemp,pctestind,pctestocc,pctprojind,pctprojocc,nchg,pchg,growrate,aopeng,aopenr,aopent,suppress
0,CO,Colorado,8,1,State,0,Colorado,33,"Colorado Long-Term Projections, 2023-2033, Jul...",1,2,0,Total All Industries,2,0,"Total, All Occupations",3148421,3638718,100.00,100.0,100.00,100.0,490297,15.5728,1.4578,NaN,NaN,NaN,0
240,CO,Colorado,8,1,State,0,Colorado,33,"Colorado Long-Term Projections, 2023-2033, Jul...",1,2,0,Total All Industries,2,110000,Management Occupations,197940,233540,6.29,100.0,6.42,100.0,35600,17.9852,1.6676,NaN,NaN,NaN,0
241,CO,Colorado,8,1,State,0,Colorado,33,"Colorado Long-Term Projections, 2023-2033, Jul...",1,2,0,Total All Industries,2,111000,Top Executives,52434,60989,1.67,100.0,1.68,100.0,8555,16.3157,1.5229,NaN,NaN,NaN,0
351,CO,Colorado,8,1,State,0,Colorado,33,"Colorado Long-Term Projections, 2023-2033, Jul...",1,2,0,Total All Industries,2,111011,Chief Executives,1140,1131,0.04,100.0,0.03,100.0,-9,-0.7895,-0.0792,NaN,NaN,NaN,0
352,CO,Colorado,8,1,State,0,Colorado,33,"Colorado Long-Term Projections, 2023-2033, Jul...",1,2,0,Total All Industries,2,111021,General and Operations Managers,50674,59132,1.61,100.0,1.63,100.0,8458,16.6910,1.5556,NaN,NaN,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
91326,CO,Colorado,8,75,Balance of State,80003,Northwest Colorado,33,"Colorado Long-Term Projections, 2023-2033, Jul...",1,2,0,Total All Industries,2,537064,"Packers and Packagers, Hand",165,173,0.11,100.0,0.10,100.0,8,4.8485,0.4746,NaN,NaN,NaN,0
91335,CO,Colorado,8,75,Balance of State,80003,Northwest Colorado,33,"Colorado Long-Term Projections, 2023-2033, Jul...",1,2,0,Total All Industries,2,537065,Stockers and Order Fillers,1254,1603,0.84,100.0,0.93,100.0,349,27.8309,2.4858,NaN,NaN,NaN,0
91347,CO,Colorado,8,75,Balance of State,80003,Northwest Colorado,33,"Colorado Long-Term Projections, 2023-2033, Jul...",1,2,0,Total All Industries,2,537081,Refuse and Recyclable Material Collectors,204,224,0.14,100.0,0.13,100.0,20,9.8039,0.9396,NaN,NaN,NaN,0
91356,CO,Colorado,8,75,Balance of State,80003,Northwest Colorado,33,"Colorado Long-Term Projections, 2023-2033, Jul...",1,2,0,Total All Industries,2,537073,Wellhead Pumpers,103,137,0.07,100.0,0.08,100.0,34,33.0097,2.8936,NaN,NaN,NaN,0


In [30]:
long["perioddesc"].value_counts()

perioddesc
Colorado Long-Term Projections, 2023-2033, July 2024    91372
Name: count, dtype: int64

In [31]:
longCIM["perioddesc"].value_counts()


perioddesc
Colorado Long-Term Projections, 2022-2032, July 2023    96797
Name: count, dtype: int64

In [35]:
long["areaname"].value_counts()

areaname
Colorado                         17623
Pueblo MSA                       15698
Denver - Aurora MSA              13652
Colorado Springs MSA              6871
Boulder-Longmont MSA              5982
Fort Collins-Loveland MSA         5891
Northwest Colorado                5817
Southwest Colorado                5518
Greeley MSA                       5161
Eastern and Southern Colorado     4875
Grand Junction MSA                4284
Name: count, dtype: int64

In [51]:
pueb = long.loc[long["areaname"] == "Pueblo MSA"]
puebCIM = longCIM.loc[longCIM["areaname"] == "Pueblo MSA"]


In [39]:
pueb.shape

(15698, 29)

In [46]:
pueb.columns

Index(['stateabbrv', 'statename', 'stfips', 'areatype', 'areatyname', 'area',
       'areaname', 'periodid', 'perioddesc', 'periodtype', 'matincodty',
       'matincode', 'matintitle', 'matoccodty', 'matoccode', 'matocctitl',
       'estemp', 'projemp', 'pctestind', 'pctestocc', 'pctprojind',
       'pctprojocc', 'nchg', 'pchg', 'growrate', 'aopeng', 'aopenr', 'aopent',
       'suppress'],
      dtype='object')

In [53]:
inew = (pueb["matocctitl"].value_counts().to_dict())
icim = puebCIM["matocctitl"].value_counts().to_dict()


In [62]:
print(pueb.shape)
print(puebCIM.shape)


(15698, 29)
(4532, 29)


In [ ]:
pueb-puebCIM

In [63]:
col= "periodtype"
inew = (pueb[col].value_counts().to_dict())
icim = puebCIM[col].value_counts().to_dict()


tot=0
for col,cnt in inew.items():
    if col not in icim:
        print(col,cnt)
        tot+=cnt

print("TOTAL ",tot)

TOTAL  0


In [64]:
icim

{1: 4532}

In [45]:
pueb.drop_duplicates().shape

(15698, 29)

In [27]:
for col,cntcim in icim.items():
    if col in inew:
        cntnew = inew[col]
    else:
        cntnew=-1
    print(f"{col:50s}  {cntnew:5d}  {cntcim:5d}  {cntnew-cntcim:5d}")

Self Employed and Unpaid Family Workers, All Jobs    8818   9088   -270
Total All Industries                                 6709   6295    414
Services Providing                                   4701   4647     54
Total Self Employed and Unpaid Family Workers, All Jobs   4409   4544   -135
Self Employed Workers, All Jobs                      4409   4544   -135
Government                                           5017   4442    575
Unclassified                                         3324   3320      4
Local Government, Excluding Education and Hospitals   3668   3126    542
Educational Services                                 1338   2706  -1368
Manufacturing                                        2094   2609   -515
Goods Producing                                      2638   2553     85
Other Services (except Government)                   2272   2316    -44
Professional, Scientific, and Technical Services     1151   2155  -1004
Education and Health Services                        1415 

## Short-Term Employment Projections in Colorado (shproj)

In [28]:
path = "/home/joe/bic_etl/cdle/cdle_LaborAndEmployment/data_transformed/"
file="shproj.csv"
file=file.strip()
short = pd.read_csv(f"{path}0824/{file}")

In [44]:
short.shape

(91198, 29)

In [45]:
short.isna().sum()

stateabbrv        0
statename         0
stfips            0
areatype          0
areatyname        0
area              0
areaname          0
periodid          0
perioddesc        0
periodtype        0
matincodty        0
matincode         0
matintitle        0
matoccodty        0
matoccode         0
matocctitl        0
estemp            0
projemp           0
pctestind         0
pctestocc         0
pctprojind        0
pctprojocc        0
nchg              0
pchg              0
growrate          0
aopeng        91198
aopenr        91198
aopent        91198
suppress          0
dtype: int64

In [17]:
short.head()

,stateabbrv,statename,stfips,areatype,areatyname,area,areaname,periodid,perioddesc,periodtype,matincodty,matincode,matintitle,matoccodty,matoccode,matocctitl,estemp,projemp,pctestind,pctestocc,pctprojind,pctprojocc,nchg,pchg,growrate,aopeng,aopenr,aopent,suppress
0,CO,Colorado,8,75,Balance of State,80002,Southwest Colorado,82,"Colorado Short-Term Projections, 2023-2025, Ma...",1,2,722000,Food Services and Drinking Places,2,0,"Total, All Occupations",10465,10767,100.0,9.11,100.0,9.09,302,2.8858,1.4326,NaN,NaN,NaN,0
1,CO,Colorado,8,21,Metropolitan Statistical Area,19740,Denver - Aurora MSA,82,"Colorado Short-Term Projections, 2023-2025, Ma...",1,2,713000,"Amusement, Gambling, and Recreation Industries",2,0,"Total, All Occupations",22150,22986,100.0,1.26,100.0,1.28,836,3.7743,1.8697,NaN,NaN,NaN,0
2,CO,Colorado,8,1,State,0,Colorado,82,"Colorado Short-Term Projections, 2023-2025, Ma...",1,2,312000,Beverage and Tobacco Product Manufacturing,2,0,"Total, All Occupations",9807,10112,100.0,0.31,100.0,0.31,305,3.1100,1.5431,NaN,NaN,NaN,0
3,CO,Colorado,8,75,Balance of State,80001,Eastern and Southern Colorado,82,"Colorado Short-Term Projections, 2023-2025, Ma...",1,2,999100,"Federal Government, Excluding Post Office",2,0,"Total, All Occupations",561,552,100.0,0.71,100.0,0.69,-9,-1.6043,-0.8054,NaN,NaN,NaN,0
4,CO,Colorado,8,75,Balance of State,80002,Southwest Colorado,82,"Colorado Short-Term Projections, 2023-2025, Ma...",1,2,813000,"Religious, Grantmaking, Civic, Professional, a...",2,0,"Total, All Occupations",969,1000,100.0,0.84,100.0,0.84,31,3.1992,1.5870,NaN,NaN,NaN,0


In [18]:
short["perioddesc"].value_counts()

perioddesc
Colorado Short-Term Projections, 2023-2025, March 2024    91198
Name: count, dtype: int64

In [20]:
shortInvt = short["areaname"].value_counts().to_dict()

In [22]:
wfile=f"https://data.colorado.gov/resource/u2t6-bfhr.csv?$limit=10000000"
shortCIM = pd.read_csv(wfile)

In [23]:
shortCIMInvt = shortCIM["areaname"].value_counts().to_dict()


In [24]:
for area,cnt in shortInvt.items():
    if area in shortCIMInvt:
        cntCIM = shortCIMInvt[area]
    else:
        cntCIM=0
    diff=cnt-cntCIM
    print(f"{area:30s} {cntCIM:5d}  {cnt:5d}  {diff:5d}")

Colorado                       17617  17617      0
Pueblo MSA                     15640  15640      0
Denver - Aurora MSA            13639  13639      0
Colorado Springs MSA            6848   6848      0
Boulder-Longmont MSA            5962   5962      0
Fort Collins-Loveland MSA       5901   5901      0
Northwest Colorado              5819   5819      0
Southwest Colorado              5509   5509      0
Greeley MSA                     5122   5122      0
Eastern and Southern Colorado   4854   4854      0
Grand Junction MSA              4287   4287      0


#  SUMMARY STATS

## Get CIM Dataset Info

In [114]:
cim_url_query = 'data.colorado.gov'
cimDatasets = {}
bicHome = "/home/joe/bic_etl"
with Socrata(cim_url_query, None) as client:
    datasets = client.datasets()
    for dataset in datasets:
        if dataset['owner']['display_name'] == 'Colorado Information Marketplace':
           title=dataset["resource"]["name"]
           w4x4=dataset["resource"]["id"]
        cimDatasets[title] = dataset


In [42]:
labforceTitle = "Unemployment Estimates in Colorado"
industryTitle = "Employee Counts by Industry in Colorado"
cesTitle = "Hours Worked by Employees in Colorado"
shortTitle = "Short-Term Employment Projections in Colorado"
incomeTitle = "Personal Income in Colorado"
oesTitle = "Employment Wages in Colorado"
longTitle = "Long-Term Employment Projections in Colorado"

## Data Type and Missing Data Stats 

In [120]:

def analyzeDtypes(df,title,datasets):
    print(f'{title}  - {datasets[title]["resource"]["id"]}')
    dfD = df.dtypes.to_dict()
    map = {"object":"Text","int64":"Number","float64":"Number"}
    dfDtypes={}
    hist={}
    for col,typ in dfD.items():
        dfDtypes[col]={}
        dfDtypes[col]["Pandas"]=str(typ)
        dfDtypes[col]["Mapped"]=map[str(typ)]
    ngood=0
    nbad=0
    print("------------------------------\nMISMATCHED COLUMNS DATA TYPES")
    for nn,name in enumerate(datasets[title]["resource"]["columns_name"]):
        cimTyp= datasets[title]["resource"]["columns_datatype"][nn]
        panTyp = dfDtypes[name]['Mapped']
        if cimTyp not in hist:
            hist[cimTyp]=0
        hist[cimTyp]+=1
        dsc = datasets[title]["resource"]["columns_description"][nn]
        if cimTyp != panTyp:
            nbad+=1
            print(f"{nn}  {name:25.25s}  CIM:{cimTyp:12.12s}  PANDAS:{dfDtypes[name]['Mapped']}     {dsc}")
        else:
            ngood+=1
    print(f"\n{nbad} Mismatches found")
    print("-------------------------------------------")
    print("Total Good Titles: ",ngood)
    for typ,cnt in hist.items():
        print(f"{typ:15.15s}   {cnt:4d}")
    print("------------------------------------------") 
    print("MISSING VALUES")
    dfMiss = df.isna().sum().to_dict()
    dfNrecs = df.shape[0]
    dfNcols = df.shape[1]
    for col,cnt in dfMiss.items():
        pp = 100*cnt/dfNrecs
        if cnt > 0:
           print(f"{col:25.25s}  {cnt:8d}  {pp:5.2f}") 
    print("-------------------------------------------")
    print("Shape: ",df.shape)
    print("Last Data Update: ",datasets[title]["resource"]["data_updated_at"])


labforceTitle = "Unemployment Estimates in Colorado"
industryTitle = "Employee Counts by Industry in Colorado"
cesTitle = "Hours Worked by Employees in Colorado"
shortTitle = "Short-Term Employment Projections in Colorado"
incomeTitle = "Personal Income in Colorado"
oesTitle = "Employment Wages in Colorado"
longTitle = "Long-Term Employment Projections in Colorado"

#analyzeDtypes(industry,industryTitle,cimDatasets)
#analyzeDtypes(labf,labforceTitle,cimDatasets)
#analyzeDtypes(ces,cesTitle,cimDatasets)
#analyzeDtypes(short,shortTitle,cimDatasets)
#analyzeDtypes(income,incomeTitle,cimDatasets)
analyzeDtypes(oes,oesTitle,cimDatasets)

#analyzeDtypes(long,longTitle,cimDatasets)


Employment Wages in Colorado  - busm-qa5b
------------------------------
MISMATCHED COLUMNS DATA TYPES
39  panelcode                  CIM:Text          PANDAS:Number     Reference panel code (yyyymm)

1 Mismatches found
-------------------------------------------
Total Good Titles:  39
Text                12
Number              28
------------------------------------------
MISSING VALUES
empcount                        545   0.17
response                         70   0.02
mean                           8248   2.50
entrywg                        8739   2.65
experience                     9212   2.80
pct10                          8338   2.53
pct25                          8760   2.66
median                         9701   2.94
pct75                         11169   3.39
pct90                         16084   4.88
udpct                         47374  14.38
udpctwage                    320890  97.41
udrnglopct                    47374  14.38
udrnghipct                    47374  14.38
udrngme

# School Programs 

In [15]:
dfSchoolMan = pd.read_csv("/home/joe/bic_etl/cdle/cdle_education/data_transformed/school_programs_final.csv")


In [16]:
dfSchoolMan.head()

,statename,city,address,areaname,areadesc,instname1,progtitle,progdesc,instowndes,insttydesc,latitude,longitude,url,zip
0,Colorado,Alamosa,208 Edgemont Blvd,Alamosa County ...,Alamosa County is one of the 64 counties of th...,Adams State University,American Government and Politics (United States).,A program that focuses on the systematic study...,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0
1,Colorado,Alamosa,208 Edgemont Blvd,Alamosa County ...,Alamosa County is one of the 64 counties of th...,Adams State University,"Biology/Biological Sciences, General.",A general program of biology at the introducto...,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0
2,Colorado,Alamosa,208 Edgemont Blvd,Alamosa County ...,Alamosa County is one of the 64 counties of th...,Adams State University,"Business Administration and Management, General.",A program that generally prepares individuals ...,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0
3,Colorado,Alamosa,208 Edgemont Blvd,Alamosa County ...,Alamosa County is one of the 64 counties of th...,Adams State University,"Business Administration, Management and Operat...",Any instructional program in business and admi...,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0
4,Colorado,Alamosa,208 Edgemont Blvd,Alamosa County ...,Alamosa County is one of the 64 counties of th...,Adams State University,"Business/Commerce, General.",A program that focuses on the general study of...,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0


In [17]:
dfSchoolMan["instname1"].value_counts()

instname1
University of Colorado Denver/Anschutz Medical Campus               850
University of Colorado Boulder                                      342
Front Range Community College                                       294
Arapahoe Community College                                          255
Westwood College-Denver North                                       196
DeVry University-Colorado                                           196
Community College of Aurora                                         175
Pickens Technical College                                           170
ITT Technical Institute-Westminster                                 161
Colorado Technical University-Greenwood Village                     150
Colorado State University-Fort Collins                              143
Aspen University                                                    141
National American University-Centennial                             130
University of Denver                                  

In [18]:
dfSchoolMan["latitude"].value_counts()

latitude
0.000000     6073
39.830429     196
39.910000     126
39.721310      66
39.551643      60
39.720000      30
39.840000      28
39.574654      20
38.913955      13
38.774820       7
38.919266       5
38.994577       4
40.060000       3
39.814965       1
39.741781       1
39.746542       1
39.859891       1
Name: count, dtype: int64

In [19]:
dfSchoolC = pd.read_csv("https://data.colorado.gov/resource/jnj7-fw37.csv?$limit=10000000")

In [20]:
dfSchoolC["latitude"].value_counts()

latitude
0.000000     6073
39.830429     196
39.910000     126
39.721310      66
39.551643      60
39.720000      30
39.840000      28
39.574654      20
38.913955      13
38.774820       7
38.919266       5
38.994577       4
40.060000       3
39.741781       1
39.814965       1
39.746542       1
39.859891       1
Name: count, dtype: int64

In [21]:
dfSchoolC.head()

,statename,city,address,areaname,areadesc,instname1,progtitle,progdesc,instowndes,insttydesc,latitude,longitude,url,zip,georeference
0,Colorado,Alamosa,208 Edgemont Blvd,Alamosa County,Alamosa County is one of the 64 counties of th...,Adams State University,American Government and Politics (United States).,A program that focuses on the systematic study...,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0,POINT (-105.880160009 37.470384015)
1,Colorado,Alamosa,208 Edgemont Blvd,Alamosa County,Alamosa County is one of the 64 counties of th...,Adams State University,"Dramatic/Theatre Arts and Stagecraft, Other.",Any instructional program in dramatic/theatre ...,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0,POINT (-105.880160009 37.470384015)
2,Colorado,Alamosa,208 Edgemont Blvd,Alamosa County,Alamosa County is one of the 64 counties of th...,Adams State University,Music Teacher Education.,A program that prepares individuals to teach m...,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0,POINT (-105.880160009 37.470384015)
3,Colorado,Alamosa,208 Edgemont Blvd,Co Rural Workforce Consortium,NaN,Adams State University,American Government and Politics (United States).,A program that focuses on the systematic study...,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0,POINT (-105.880160009 37.470384015)
4,Colorado,Alamosa,208 Edgemont Blvd,Edwards MCA,Eagle & Lake counties,Adams State University,Music Teacher Education.,A program that prepares individuals to teach m...,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0,POINT (-105.880160009 37.470384015)


### Fix School Data

In [42]:
dfSchoolMan = pd.read_csv("/home/joe/bic_etl/cdle/cdle_education/data_transformed/school_programs.csv")


In [43]:
dfSchoolMan.shape

(10304, 13)

In [44]:
dfbad = dfSchoolMan.loc[dfSchoolMan['progdesc'].isna()]
dfgood = dfSchoolMan.loc[~dfSchoolMan['progdesc'].isna()]

In [45]:
print(dfbad.shape)
print(dfgood.shape)


(5542, 13)
(4762, 13)


In [50]:
tot=0
histbad = {}
dftmp = pd.DataFrame(columns=dfgood.columns)
dfdel = pd.DataFrame(columns=dfgood.columns)

for index,row in dfbad.iterrows():
    inst = row['instname1']
    prog = row['progtitle']
    area = row['areaname']
    url = row['url']
    ownd = row['instowndes']
    indes = row['insttydesc']
    if inst not in histbad:
        histbad[inst]={}
        histbad[inst]['count']=0
        histbad[inst]['bad']=0
        
    histbad[inst]['count']+=1
    nn = dfgood.loc[(dfgood['instname1'] == inst) & 
                    (dfgood['progtitle'] == prog) & 
                    (dfgood['areaname']==area) & 
                    (dfgood['url'] == url) & 
                    (dfgood['instowndes'] == ownd) & 
                    (dfgood['insttydesc'] == indes)                   
                   ].shape[0]
    if nn == 0: # There are no other records, so we need to keep this record
        tot+=1
        histbad[inst]['bad']+=1
    #    dftmp = dftmp.append(row)
        dftmp=pd.concat([dftmp,row.to_frame().T])
    elif nn == 1: # There IS another record and the descripiton is NOT missing, so we can delete this record with the missing description
    #    dfdel=dfdel.append(row)
        dfdel=pd.concat([dfdel,row.to_frame().T])
    else:
        print(inst,prog,area)
        
        
print(dfbad.shape[0],tot)
  

5542 1873


In [53]:
print(dftmp.shape)
print(dfdel.shape)

(1873, 13)
(3669, 13)


In [7]:
dfdel.shape

(3669, 14)

In [ ]:
for inst,res in histbad.items():
    print(inst,res['count'],res['bad'])

In [55]:
dfdel.head()

,statename,city,areaname,areadesc,instname1,progtitle,progdesc,instowndes,insttydesc,latitude,longitude,url,zip
85,"""Colorado""","""Alamosa""","""Alamosa County ...","""Alamosa County is one of the 64 counties of t...","""Adams State University""","""American Government and Politics (United Stat...",NaN,"""Public Institution""","""Four-year Colleges and Universities""","""0.000000""","""0.000000""","""http://www.adams.edu""","""81101"""
87,"""Colorado""","""Alamosa""","""Alamosa County ...","""Alamosa County is one of the 64 counties of t...","""Adams State University""","""Biology/Biological Sciences, General.""",NaN,"""Public Institution""","""Four-year Colleges and Universities""","""0.000000""","""0.000000""","""http://www.adams.edu""","""81101"""
89,"""Colorado""","""Alamosa""","""Alamosa County ...","""Alamosa County is one of the 64 counties of t...","""Adams State University""","""Business Administration and Management, Gener...",NaN,"""Public Institution""","""Four-year Colleges and Universities""","""0.000000""","""0.000000""","""http://www.adams.edu""","""81101"""
91,"""Colorado""","""Alamosa""","""Alamosa County ...","""Alamosa County is one of the 64 counties of t...","""Adams State University""","""Business Administration, Management and Opera...",NaN,"""Public Institution""","""Four-year Colleges and Universities""","""0.000000""","""0.000000""","""http://www.adams.edu""","""81101"""
93,"""Colorado""","""Alamosa""","""Alamosa County ...","""Alamosa County is one of the 64 counties of t...","""Adams State University""","""Business/Commerce, General.""",NaN,"""Public Institution""","""Four-year Colleges and Universities""","""0.000000""","""0.000000""","""http://www.adams.edu""","""81101"""


In [35]:
4762+1873

6635

In [36]:
5542-1873

3669

In [56]:
## Add the records with missing descriptions that do not have a corresponding non-missing description record back for output 
dfgood = pd.concat([dfgood,dftmp])

In [57]:
for index,row in dfdel.iterrows():
    inst = row['instname1']
    prog = row['progtitle']
    area = row['areaname']
    url = row['url']
    ownd = row['instowndes']
    indes = row['insttydesc']
 
    histbad[inst]['count']+=1
    hld= dfgood.loc[(dfgood['instname1'] == inst) & 
                    (dfgood['progtitle'] == prog) & 
                    (dfgood['areaname']==area) & 
                    (dfgood['url'] == url) & 
                    (dfgood['instowndes'] == ownd) & 
                    (dfgood['insttydesc'] == indes)                   
                   ]
    nn=hld.shape[0]
    
    if nn == 1:
        mm = hld['progdesc'].isna().sum()
        if mm != 0:
            print("ROH ROH")
            print(hld.head())
            print("----")
    else:
        print("BAD BAD BAD",nn)
        print(hld.head())
        print("----")

In [58]:
dfgood.to_csv("school_programs.csv",index=False)

In [7]:
dfschoolNew = pd.read_csv("school_programs.csv")

In [8]:
dfschoolNew.shape

(6635, 13)

In [9]:
dfSchoolC = pd.read_csv("https://data.colorado.gov/resource/jnj7-fw37.csv?$limit=10000000")

In [5]:
dfSchoolC.shape

(6635, 15)

In [11]:
dfschoolNew.shape

(6635, 13)

In [12]:
upt = dfschoolNew["instname1"].value_counts().to_dict()
cim = dfSchoolC["instname1"].value_counts().to_dict()


In [14]:
for ins,cnt in upt.items():
    ins=ins.strip()
    ins=ins.replace('"',"")
 #   print(ins,cnt)
 #   print(len(ins),ins)
    if ins in cim:
       cntCIM=cim[ins]
       diff=cnt-cntCIM
       print(f"{ins:65s}  {cntCIM:3d}  {cnt:3d}  {diff:3d}")
    else:
       print("ROH ROH ROH ",ins)

University of Colorado Denver/Anschutz Medical Campus              850  850    0
University of Colorado Boulder                                     342  342    0
Front Range Community College                                      294  294    0
Arapahoe Community College                                         255  255    0
Westwood College-Denver North                                      196  196    0
DeVry University-Colorado                                          196  196    0
Community College of Aurora                                        175  175    0
Pickens Technical College                                          170  170    0
ITT Technical Institute-Westminster                                161  161    0
Colorado Technical University-Greenwood Village                    150  150    0
Colorado State University-Fort Collins                             143  143    0
Aspen University                                                   141  141    0
National American University

## Xrefs

In [4]:
xrefsTitles= {
"2cpa-vbur":"Income-Data-for-Colorado-Counties",
"busm-qa5b":"Occupational-Employment-Statistics",
"cjkq-q9ih ":"Employment-by-Industry-from-Census-of-Employment",
"4e3w-qire":"Employment-and-Unemployment-Estimates",
"pt2g-89wc":"Current-Employment-Statistics",
"gyeb-jc69":"Long-Term Employment Projections in Colorado",
"u2t6-bfhr":"Short-Term Employment Projections in Colorado"}

xrefsFiles = {
"2cpa-vbur":"Income.csv",
"busm-qa5b":"OESWage.csv",
"cjkq-q9ih":"Industry.csv",
"4e3w-qire":"Labforce.csv",
"pt2g-89wc":"Ces.csv",
"gyeb-jc69":"lngproj.csv",
"u2t6-bfhr":"shproj.csv"}

xrefs4x4s = {val:key for key,val in xrefsFiles.items()}

# OLD STUFF

In [2]:
path = "/home/joe/bic_etl/cdle/cdle_LaborAndEmployment/data_transformed/"
path=""
file="lngproj.csv"
file=file.strip()
#long = pd.read_csv(f"{path}0824/{file}")
longN = pd.read_csv(f"{file}")

In [3]:
wfile=f"https://data.colorado.gov/resource/gyeb-jc69.csv?$limit=10000000"
longCIM = pd.read_csv(wfile)

In [4]:
path = "/home/joe/bic_etl/cdle/cdle_LaborAndEmployment/data_transformed/"

file="lngproj.csv"
file=file.strip()
longO = pd.read_csv(f"{path}0424/{file}")
#long = pd.read_csv(f"{file}")

In [5]:
print(longCIM.shape)
print(longN.shape)
print(longO.shape)


(96797, 29)
(91372, 29)
(96797, 29)


In [6]:
puebN = longN.loc[longN["areaname"] == "Pueblo MSA"]
puebC = longCIM.loc[longCIM["areaname"] == "Pueblo MSA"]


In [7]:
print(puebN.shape)
print(puebC.shape)


(15698, 29)
(4532, 29)


In [13]:
puebN.columns

Index(['stateabbrv', 'statename', 'stfips', 'areatype', 'areatyname', 'area',
       'areaname', 'periodid', 'perioddesc', 'periodtype', 'matincodty',
       'matincode', 'matintitle', 'matoccodty', 'matoccode', 'matocctitl',
       'estemp', 'projemp', 'pctestind', 'pctestocc', 'pctprojind',
       'pctprojocc', 'nchg', 'pchg', 'growrate', 'aopeng', 'aopenr', 'aopent',
       'suppress'],
      dtype='object')

In [16]:
print(puebN.loc[puebN["matocctitl"] == "Interpreters and Translators","matintitle"].value_counts())
print(puebC.loc[puebC["matocctitl"] == "Interpreters and Translators","matintitle"].value_counts())

matintitle
State Government, Excluding Education and Hospitals        2
Government                                                 2
Professional, Scientific, and Technical Services           2
Educational Services                                       2
Self Employed and Unpaid Family Workers, All Jobs          2
Social Assistance                                          1
Self Employed Workers, All Jobs                            1
Services Providing                                         1
Education and Health Services                              1
Health Care and Social Assistance                          1
Professional and Business Services                         1
Total All Industries                                       1
Unclassified                                               1
Total Self Employed and Unpaid Family Workers, All Jobs    1
Name: count, dtype: int64
matintitle
Self Employed and Unpaid Family Workers, All Jobs          2
Total Self Employed and Unpaid Family

In [18]:
aN = puebN["matintitle"].value_counts().to_dict()
aC = puebC["matintitle"].value_counts().to_dict()

In [22]:
tot=0
for key in aN.keys():
    cntN = aN[key]
    if key in aC:
        cntA = aC[key]
    else:
        cntA=0
    tot+=cntN-cntA
    print(f"{key:50s}  {cntA:4d}  {cntN:4d}  {cntN-cntA:4d}")

Self Employed and Unpaid Family Workers, All Jobs    566   856   290
Services Providing                                   274   690   416
Government                                           211   663   452
Educational Services                                 120   546   426
Local Government, Excluding Education and Hospitals   116   495   379
Professional, Scientific, and Technical Services      68   486   418
Total All Industries                                 418   475    57
Manufacturing                                         92   473   381
Other Services (except Government)                   112   468   356
Construction                                          90   429   339
Total Self Employed and Unpaid Family Workers, All Jobs   283   428   145
Self Employed Workers, All Jobs                      283   428   145
Unclassified                                         206   403   197
Education and Health Services                        101   390   289
State Government, Excluding 

In [23]:
tot

11278

In [12]:
hist={}
for col in puebN.columns:
    N = puebN[col].value_counts().to_dict()
    C = puebC[col].value_counts().to_dict()
    print(col)
    tot=0
    for key in N.keys():
        cntN = N[key]
        if C.get(key):
           cntC = C[key]
        else:
           cntC=0
        key=str(key)
        print(f"    {key:20s}  {cntC:6d}  {cntN:6d} {cntN-cntC:5d}")
        tot+=cntN-cntC
    hist[col]=tot   

stateabbrv
    CO                      4532   15698 11166
statename
    Colorado                4532   15698 11166
stfips
    8                       4532   15698 11166
areatype
    21                      4532   15698 11166
areatyname
    Metropolitan Statistical Area    4532   15698 11166
area
    39380                   4532   15698 11166
areaname
    Pueblo MSA              4532   15698 11166
periodid
    33                         0   15698 15698
perioddesc
    Colorado Long-Term Projections, 2023-2033, July 2024       0   15698 15698
periodtype
    1                       4532   15698 11166
matincodty
    2                       4532   15698 11166
matincode
    102000                   274     690   416
    0                        418     475    57
    6010                     283     428   145
    67                       283     428   145
    670                      283     428   145
    671                      283     428   145
    102900                   206     403   197

In [11]:
hist

{'stateabbrv': 11166,
 'statename': 11166,
 'stfips': 11166,
 'areatype': 11166,
 'areatyname': 11166,
 'area': 11166,
 'areaname': 11166,
 'periodid': 15698,
 'perioddesc': 15698,
 'periodtype': 11166,
 'matincodty': 11166,
 'matincode': 11241,
 'matintitle': 11278,
 'matoccodty': 11166,
 'matoccode': 11191,
 'matocctitl': 11223,
 'estemp': 11721,
 'projemp': 11726,
 'pctestind': 11646,
 'pctestocc': 12312,
 'pctprojind': 11640,
 'pctprojocc': 12279,
 'nchg': 11412,
 'pchg': 12857,
 'growrate': 12840,
 'aopeng': 0,
 'aopenr': 0,
 'aopent': 0,
 'suppress': 11166}

In [2]:
path = "/home/joe/bic_etl/cdle/cdle_LaborAndEmployment/data_transformed/"

file="lngproj.csv"
file=file.strip()
longO3 = pd.read_csv(f"{path}0324/{file}")
#long = pd.read_csv(f"{file}")

In [3]:
longO3.shape

(96797, 29)

In [4]:
longO3.columns

Index(['stateabbrv', 'statename', 'stfips', 'areatype', 'areatyname', 'area',
       'areaname', 'periodid', 'perioddesc', 'periodtype', 'matincodty',
       'matincode', 'matintitle', 'matoccodty', 'matoccode', 'matocctitl',
       'estemp', 'projemp', 'pctestind', 'pctestocc', 'pctprojind',
       'pctprojocc', 'nchg', 'pchg', 'growrate', 'aopeng', 'aopenr', 'aopent',
       'suppress'],
      dtype='object')

In [5]:
longO3["perioddesc"].value_counts()

perioddesc
Colorado Long-Term Projections, 2022-2032, July 2023    96797
Name: count, dtype: int64

In [8]:
longO["perioddesc"].value_counts()


perioddesc
Colorado Long-Term Projections, 2022-2032, July 2023    96797
Name: count, dtype: int64

In [11]:
longCIM["perioddesc"].value_counts()

perioddesc
Colorado Long-Term Projections, 2022-2032, July 2023    96797
Name: count, dtype: int64

In [12]:
longN["perioddesc"].value_counts()


perioddesc
Colorado Long-Term Projections, 2023-2033, July 2024    91372
Name: count, dtype: int64

## Read CIM Data

In [17]:
dfs = {}
for file in files:
    file=file.strip()
    w4x4=xrefs4x4s[file]
    wfile=f"https://data.colorado.gov/resource/{w4x4}.csv?$limit=10000000"
    print(file,w4x4,wfile)
    
    df = pd.read_csv(wfile)
    dfs[file]=df

NameError: name 'files' is not defined

In [7]:
for file,df in dfs.items():
    print(file,df.columns)

Ces.csv Index(['stateabbrv', 'statename', 'stfips', 'areatyname', 'areaname', 'area',
       'periodyear', 'periodtype', 'pertypdesc', 'period', 'seriescode',
       'seriesttls', 'seriesdesc', 'adjusted', 'benchmark', 'prelim', 'empces',
       'empprodwrk', 'empfemale', 'hours', 'earnings', 'hourearn',
       'supprecord', 'supphe', 'supppw', 'suppfem', 'hoursallwrkr',
       'earningsallwrkr', 'hourearnallwrkr', 'suppheallwrkr'],
      dtype='object')
Income.csv Index(['stateabbrv', 'statename', 'stfips', 'areatyname', 'areaname',
       'areatype', 'area', 'periodyear', 'periodtype', 'pertypdesc', 'period',
       'inctype', 'incdesc', 'incsource', 'incsrcdesc', 'income', 'incrank',
       'population', 'releasedate'],
      dtype='object')
Industry.csv Index(['stateabbrv', 'statename', 'stfips', 'areaname', 'areatype',
       'areatyname', 'area', 'periodyear', 'periodtype', 'period',
       'pertypdesc', 'indcodty', 'indcode', 'codetitle', 'ownership',
       'ownertitle', 'preli

In [17]:
print(dfs['shproj.csv']['perioddesc'].value_counts())
print(dfs3['shproj.csv']['perioddesc'].value_counts())
print(dfs4['shproj.csv']['perioddesc'].value_counts())

Colorado Short-Term Projections, 2016-2018, June 2016        372870
Colorado Short-Term Projections, 2020-2022, December 2019     86350
Name: perioddesc, dtype: int64
Colorado Short-Term Projections, 2016-2018, June 2016    74574
Name: perioddesc, dtype: int64
Colorado Short-Term Projections, 2016-2018, June 2016    74574
Name: perioddesc, dtype: int64


    CIM
    Colorado Short-Term Projections, 2016-2018, June 2016        372870
    Colorado Short-Term Projections, 2020-2022, December 2019     86350

    0324 We Pulled
    Colorado Short-Term Projections, 2016-2018, June 2016    74574

    0424 We Pulled
    Colorado Short-Term Projections, 2016-2018, June 2016    74574

    0324 Actual
    69	Colorado Short-Term Projections, 2016-2018, June 2016
    81	Colorado Short-Term Projections, 2022-2024, July 2023
    82	Colorado Short-Term Projections, 2023-2025, March 2024

    0424 Actual
    69	Colorado Short-Term Projections, 2016-2018, June 2016
    82	Colorado Short-Term Projections, 2023-2025, March 2024

In [11]:
dfs['lngproj.csv']['perioddesc'].value_counts()

Colorado Long-Term Projections, 2022-2032, July 2023    193594
Colorado Long-Term Projections, 2019-2029, June 2020    170496
Colorado Long-Term Projections, 2020-2030, July 2021     90055
Name: perioddesc, dtype: int64

In [16]:
print(dfs4['shproj.csv']['periodid'].value_counts())
print(dfs3['shproj.csv']['periodid'].value_counts())

69    74574
Name: periodid, dtype: int64
69    74574
Name: periodid, dtype: int64


In [8]:
ffs = ["Ces.csv","Income.csv","Industry.csv","Labforce.csv","OESWage.csv"]
for file in ffs:
    print(file)
    print(dfs[file]["periodyear"].value_counts().sort_index())
    print("---------------")

Ces.csv
1990    4431
1991    4431
1992    4431
1993    4431
1994    4431
1995    4431
1996    4431
1997    4431
1998    4431
1999    4431
2000    4457
2001    4457
2002    4470
2003    4535
2004    4561
2005    4561
2006    4561
2007    4561
2008    4561
2009    4561
2010    4561
2011    4561
2012    4561
2013    4561
2014    4561
2015    4561
2016    4561
2017    4561
2018    4561
2019    4535
2020    4535
2021    4535
2022    4535
2023    4431
2024     991
Name: periodyear, dtype: int64
---------------
Income.csv
1929      4
1930      4
1931      4
1932      4
1933      4
       ... 
2015    201
2016    197
2017    132
2020     65
2021     65
Name: periodyear, Length: 91, dtype: int64
---------------
Industry.csv
2000     60694
2001     63377
2002     64410
2003     64763
2004     64815
2005     61845
2006     64695
2007    106092
2008     53939
2009     53976
2010     52878
2011     51565
2012     51574
2013     51512
2014     51422
2015     51274
2016     51713
2017     51786
2018 

In [11]:
dfs["lngproj.csv"].head()

,stateabbrv,statename,stfips,areatype,areatyname,area,areaname,periodid,perioddesc,periodtype,matincodty,matincode,matintitle,matoccodty,matoccode,matocctitl,estemp,projemp,pctestind,pctprojind,nchg,aopeng,aopenr,aopent,suppress,pctprojocc,pctestocc,growrate,pchg
0,CO,Colorado,8,75,Balance of State,80001,Eastern and Southern Colorado,29,"Colorado Long-Term Projections, 2019-2029, Jun...",1,2,541000,"Professional, Scientific, and Technical Services",2,292000,Health Technologists and Technicians,76,86,6.95,7.60,10,NaN,NaN,NaN,0,4.32,4.01,1.2438,13.1579
1,CO,Colorado,8,21,Metropolitan Statistical Area,14500,Boulder-Longmont MSA,29,"Colorado Long-Term Projections, 2019-2029, Jun...",1,2,0,Total All Industries,2,191029,"Biological Scientists, All Other",213,265,0.10,0.12,52,NaN,NaN,NaN,0,100.00,100.00,2.2084,24.4131
2,CO,Colorado,8,21,Metropolitan Statistical Area,39380,Pueblo MSA,29,"Colorado Long-Term Projections, 2019-2029, Jun...",1,2,620000,Health Care and Social Assistance,2,110000,Management Occupations,368,422,2.70,2.74,54,NaN,NaN,NaN,0,14.62,13.77,1.3786,14.6739
3,CO,Colorado,8,1,State,0,Colorado,29,"Colorado Long-Term Projections, 2019-2029, Jun...",1,2,560000,Administrative and Support and Waste Managemen...,2,119000,Other Management Occupations,617,717,0.38,0.40,100,NaN,NaN,NaN,0,0.82,0.79,1.5134,16.2075
4,CO,Colorado,8,21,Metropolitan Statistical Area,19740,Denver - Aurora MSA,29,"Colorado Long-Term Projections, 2019-2029, Jun...",1,2,102000,Services Providing,2,193000,Social Scientists and Related Workers,2913,3395,0.21,0.23,482,NaN,NaN,NaN,0,79.81,79.85,1.5430,16.5465


In [16]:
display(dfs["lngproj.csv"]["perioddesc"].value_counts())

Colorado Long-Term Projections, 2022-2032, July 2023    193594
Colorado Long-Term Projections, 2019-2029, June 2020    170496
Colorado Long-Term Projections, 2020-2030, July 2021     90055
Name: perioddesc, dtype: int64

In [6]:
dfs["shproj.csv"].head()

,stateabbrv,statename,stfips,areatype,areatyname,areaname,periodid,perioddesc,periodtype,matincodty,matintitle,matoccodty,matocctitl,estemp,projemp,nchg,aopeng,aopenr,aopent,suppress,area,matincode,matoccode,pctestind,pctprojind,pchg,growrate,pctprojocc,pctestocc
0,CO,Colorado,8,21,Metropolitan Statistical Area,Colorado Springs MSA,76,"Colorado Short-Term Projections, 2020-2022, De...",1,2,Services Providing,2,Optometrists,98,99,1,NaN,NaN,NaN,0,17820,102000,291041,0.04,0.04,1.0204,0.5089,89.19,89.09
1,CO,Colorado,8,21,Metropolitan Statistical Area,Denver - Aurora MSA,76,"Colorado Short-Term Projections, 2020-2022, De...",1,2,"Self Employed and Unpaid Family Workers, All Jobs",2,"Shipping, Receiving, and Traffic Clerks",7,7,0,NaN,NaN,NaN,0,19740,67,435071,0.01,0.01,0.0000,0.0000,0.13,0.13
2,CO,Colorado,8,1,State,Colorado,76,"Colorado Short-Term Projections, 2020-2022, De...",1,2,Printing and Related Support Activities,2,"Installation, Maintenance, and Repair Occupations",43,40,-3,NaN,NaN,NaN,0,0,323000,490000,0.88,0.87,-6.9767,-3.5514,0.04,0.04
3,CO,Colorado,8,21,Metropolitan Statistical Area,Fort Collins-Loveland MSA,76,"Colorado Short-Term Projections, 2020-2022, De...",1,2,Finance and Insurance,2,Office and Administrative Support Occupations,1382,1381,-1,NaN,NaN,NaN,0,22660,520000,430000,37.73,37.28,-0.0724,-0.0362,6.09,6.06
4,CO,Colorado,8,75,Balance of State,Eastern and Southern Colorado,76,"Colorado Short-Term Projections, 2020-2022, De...",1,2,Government,2,Financial Specialists,194,197,3,NaN,NaN,NaN,0,80001,900000,132000,1.85,1.87,1.5464,0.7702,19.54,19.44


In [6]:
display(dfs["shproj.csv"]["perioddesc"].value_counts())
display(dfs["lngproj.csv"]["perioddesc"].value_counts())

Colorado Short-Term Projections, 2016-2018, June 2016        372870
Colorado Short-Term Projections, 2020-2022, December 2019     86350
Name: perioddesc, dtype: int64

Colorado Long-Term Projections, 2022-2032, July 2023    193594
Colorado Long-Term Projections, 2019-2029, June 2020    170496
Colorado Long-Term Projections, 2020-2030, July 2021     90055
Name: perioddesc, dtype: int64

In [79]:
for file,df in dfs.items():
    print(file,df.shape,dfs4[file].shape,dfs4[file].shape[0]-df.shape[0])
    # print(df.columns)
    # print(dfs4[file].columns)

Ces.csv (151495, 30) (154206, 30) 2711
Income.csv (8892, 19) (9022, 19) 130
Industry.csv (1346348, 29) (1372736, 29) 26388
Labforce.csv (34959, 18) (35598, 18) 639
OESWage.csv (284061, 40) (284676, 40) 615
lngproj.csv (260551, 29) (96797, 29) -163754
shproj.csv (310072, 29) (74574, 29) -235498


Ces.csv (151495, 30) (154206, 30) 2711   Replace
Income.csv (8892, 19) (9022, 19) 130     Replace
Industry.csv (1346348, 29) (1372736, 29) 26388  Replace
Labforce.csv (34959, 18) (35598, 18) 639  Replace
OESWage.csv (284061, 40) (284676, 40) 615 Replace
lngproj.csv (260551, 29) (96797, 29) -163754  Upsert
shproj.csv (310072, 29) (74574, 29) -235498  Upsert

In [8]:
dfs4.keys()

dict_keys(['Ces.csv', 'Income.csv', 'Industry.csv', 'Labforce.csv', 'OESWage.csv', 'lngproj.csv', 'shproj.csv'])

## Check Period of Record

In [11]:
dfs['Ces.csv'].columns

Index(['stateabbrv', 'statename', 'stfips', 'areatyname', 'areaname', 'area',
       'periodyear', 'periodtype', 'pertypdesc', 'period', 'seriescode',
       'seriesttls', 'seriesdesc', 'adjusted', 'benchmark', 'prelim', 'empces',
       'empprodwrk', 'empfemale', 'hours', 'earnings', 'hourearn',
       'supprecord', 'supphe', 'supppw', 'suppfem', 'hoursallwrkr',
       'earningsallwrkr', 'hourearnallwrkr', 'suppheallwrkr'],
      dtype='object')

In [15]:
print(dfs['Ces.csv']['pertypdesc'].value_counts())
print(dfs['Ces.csv']['periodtype'].value_counts())

Monthly    143404
Annual      10802
Name: pertypdesc, dtype: int64
3    143404
1     10802
Name: periodtype, dtype: int64


In [14]:
dfs['Ces.csv']['period'].value_counts()

2     12237
1     12237
12    11893
11    11893
10    11893
9     11893
8     11893
7     11893
6     11893
5     11893
4     11893
3     11893
0     10802
Name: period, dtype: int64

In [31]:
ffs = ["Ces.csv","Income.csv","Industry.csv","Labforce.csv","OESWage.csv"]
for file in ffs:
    print(file)
    print(dfs[file]['periodtype'].value_counts())
    print(dfs[file]['pertypdesc'].value_counts())
    
    

Ces.csv
3    143404
1     10802
Name: periodtype, dtype: int64
Monthly    143404
Annual      10802
Name: pertypdesc, dtype: int64
Income.csv
1    9022
Name: periodtype, dtype: int64
Annual    9022
Name: pertypdesc, dtype: int64
Industry.csv
2    1087830
1     258518
Name: periodtype, dtype: int64
Quarterly    1087830
Annual        258518
Name: pertypdesc, dtype: int64
Labforce.csv
3    33136
1     2462
Name: periodtype, dtype: int64
Monthly    33136
Annual      2462
Name: pertypdesc, dtype: int64
OESWage.csv
1    284061
Name: periodtype, dtype: int64
Annual    284061
Name: pertypdesc, dtype: int64


In [ ]:
### Labforce.csv -   [Unemployment Estimates in Colorado](https://data.colorado.gov/a/b/4e3w-qire) - "labforce"
### Industry.csv -   [Employee Counts by Industry in Colorado](https://data.colorado.gov/a/b/cjkq-q9ih) - "industry"
### Ces.csv -     [Hours Worked by Employees in Colorado](https://data.colorado.gov/a/b/pt2g-89wc) - "ces"
### Income.csv -  [Personal Income in Colorado](https://data.colorado.gov/a/b/2cpa-vbur) - "income"
### OESWage.csv - [Employment Wages in Colorado](https://data.colorado.gov/a/b/busm-qa5b) - "oes"

### lngproj.csv - [Long-Term Employment Projections in Colorado](https://data.colorado.gov/a/b/gyeb-jc69) - "lng proj, lmi"
### shproj.csv -  [Short-Term Employment Projections in Colorado](https://data.colorado.gov/a/b/u2t6-bfhr) - "shrt proj, lmi"


In [8]:
ffs = ["Ces.csv","Income.csv","Industry.csv","Labforce.csv","OESWage.csv"]
for file in ffs:
    invtYrMo={}
    print(file)
    print("Year  Jan  Feb  Mar  Apr  May  Jun  Jul  Aug  Sep  Oct  Nov  Dec")
    for index,row in dfs[file].loc[dfs[file]['periodtype']==3].iterrows():
        year=row['periodyear']
        month=row['period']

        if year not in invtYrMo:
            invtYrMo[year]={}
        if month not in invtYrMo[year]:
            invtYrMo[year][month]=0
        invtYrMo[year][month]+=1
        
    for year,invt in sorted(invtYrMo.items()):
        print(f"{year:4d}",end="")
        for mo in range(1,13):
            if mo in invt:
               cnt=invt[mo]
            else:
               cnt=0
            print(f"{cnt:5d}",end="")
        print()
    print()
    print()
        
          
        

Ces.csv
Year  Jan  Feb  Mar  Apr  May  Jun  Jul  Aug  Sep  Oct  Nov  Dec
1990  344  344  344  344  344  344  344  344  344  344  344  344
1991  344  344  344  344  344  344  344  344  344  344  344  344
1992  344  344  344  344  344  344  344  344  344  344  344  344
1993  344  344  344  344  344  344  344  344  344  344  344  344
1994  344  344  344  344  344  344  344  344  344  344  344  344
1995  344  344  344  344  344  344  344  344  344  344  344  344
1996  344  344  344  344  344  344  344  344  344  344  344  344
1997  344  344  344  344  344  344  344  344  344  344  344  344
1998  344  344  344  344  344  344  344  344  344  344  344  344
1999  344  344  344  344  344  344  344  344  344  344  344  344
2000  346  346  346  346  346  346  346  346  346  346  346  346
2001  346  346  346  346  346  346  346  346  346  346  346  346
2002  347  347  347  347  347  347  347  347  347  347  347  347
2003  352  352  352  352  352  352  352  352  352  352  352  352
2004  354  354  3

In [33]:
ffs = ["Ces.csv","Income.csv","Industry.csv","Labforce.csv","OESWage.csv"]
for file in ffs:
    invtYr={}
    print(file)
    for index,row in dfs[file].loc[dfs[file]['periodtype']==1].iterrows():
        year=row['periodyear']
        month=row['period']

        if year not in invtYr:
            invtYr[year]=0
        invtYr[year]+=1
      
    for year,cnt in sorted(invtYr.items()):
        print(f"{year:4d}  {cnt:5d}")

Ces.csv
1990    303
1991    303
1992    303
1993    303
1994    303
1995    303
1996    303
1997    303
1998    303
1999    303
2000    305
2001    305
2002    306
2003    311
2004    313
2005    313
2006    313
2007    313
2008    313
2009    313
2010    313
2011    313
2012    313
2013    313
2014    313
2015    313
2016    313
2017    313
2018    313
2019    311
2020    311
2021    311
2022    311
2023    303
2024    303
Income.csv
1929      4
1930      4
1931      4
1932      4
1933      4
1934      4
1935      4
1936      4
1937      4
1938      4
1939      4
1940      4
1941      4
1942      4
1943      4
1944      4
1945      4
1946      4
1947      4
1948      4
1949      4
1950      4
1951      4
1952      4
1953      4
1954      4
1955      4
1956      4
1957      4
1958      4
1959      4
1960      4
1961      4
1962      4
1963      4
1964      4
1965      4
1966      4
1967      4
1968      4
1969    146
1970    146
1971    146
1972    146
1973    146
1974    146
1975    1

In [23]:
for year,invt in sorted(invtYrMo.items()):
    print(f"{year:4d}",end="")
    for mo in range(1,13):
        if mo in invt:
           cnt=invt[mo]
        else:
           cnt=0
        print(f"{cnt:5d}",end="")
    print()
          
        

1990  344  344  344  344  344  344  344  344  344  344  344  344
1991  344  344  344  344  344  344  344  344  344  344  344  344
1992  344  344  344  344  344  344  344  344  344  344  344  344
1993  344  344  344  344  344  344  344  344  344  344  344  344
1994  344  344  344  344  344  344  344  344  344  344  344  344
1995  344  344  344  344  344  344  344  344  344  344  344  344
1996  344  344  344  344  344  344  344  344  344  344  344  344
1997  344  344  344  344  344  344  344  344  344  344  344  344
1998  344  344  344  344  344  344  344  344  344  344  344  344
1999  344  344  344  344  344  344  344  344  344  344  344  344
2000  346  346  346  346  346  346  346  346  346  346  346  346
2001  346  346  346  346  346  346  346  346  346  346  346  346
2002  347  347  347  347  347  347  347  347  347  347  347  347
2003  352  352  352  352  352  352  352  352  352  352  352  352
2004  354  354  354  354  354  354  354  354  354  354  354  354
2005  354  354  354  354 

## Short and Long Term Projections

In [19]:
print(dfs4['lngproj.csv']['perioddesc'].value_counts())
print(dfs3['lngproj.csv']['perioddesc'].value_counts())
print(dfs['lngproj.csv']['perioddesc'].value_counts())

Colorado Long-Term Projections, 2022-2032, July 2023    96797
Name: perioddesc, dtype: int64
Colorado Long-Term Projections, 2022-2032, July 2023    96797
Name: perioddesc, dtype: int64
Colorado Long-Term Projections, 2022-2032, July 2023    193594
Colorado Long-Term Projections, 2019-2029, June 2020    170496
Colorado Long-Term Projections, 2020-2030, July 2021     90055
Name: perioddesc, dtype: int64


### Long Term


CIM
Colorado Long-Term Projections, 2019-2029, June 2020    170496
Colorado Long-Term Projections, 2020-2030, July 2021     90055

March From CDLE
32	Colorado Long-Term Projections, 2022-2032, July 2023

April From CDLE
32	Colorado Long-Term Projections, 2022-2032, July 2023

Labels
Colorado Long-Term Projections, 2015-2025, August 2016
Colorado Long-Term Projections, 2016-2026, August 2017
Colorado Long-Term Projections, 2017-2027, June 2018
Colorado Long-Term Projections, 2021-2031, August 2022
Colorado Long-Term Projections, 2022-2032, July 2023



### Short Term


CIM
Colorado Short-Term Projections, 2016-2018, June 2016        372870
Colorado Short-Term Projections, 2020-2022, December 2019     86350

March CDLE Update
69	Colorado Short-Term Projections, 2016-2018, June 2016
81	Colorado Short-Term Projections, 2022-2024, July 2023
82	Colorado Short-Term Projections, 2023-2025, March 202404 

April CDLE Update
69	Colorado Short-Term Projections, 2016-2018, June 2016
82	Colorado Short-Term Projections, 2023-2025, March 2024

Labels
Colorado Short-Term Projections, 2016-2018, June 2016
Colorado Short-Term Projections, 2017-2019, Dec, 2016
Colorado Short-Term Projections, 2017-2019, August 2017
Colorado Short-Term Projections, 2019-2021, December 2018
Colorado Short-Term Projections, 2020-2022, December 2019
Colorado Short-Term Projections, 2022-2024, February 2023
Colorado Short-Term Projections, 2022-2024, July 2023
Colorado Short-Term Projections, 2023-2025, March 2024

In [12]:

dfs4['shproj.csv'].columns

Index(['Changed database context to 'GoCode_0424'.'], dtype='object')

In [18]:
print(dfs4['shproj.csv']['perioddesc'].value_counts())
print(dfs3['shproj.csv']['perioddesc'].value_counts())
print(dfs['shproj.csv']['perioddesc'].value_counts())

Colorado Short-Term Projections, 2023-2025, March 2024    91198
Colorado Short-Term Projections, 2016-2018, June 2016     74574
Name: perioddesc, dtype: int64
Colorado Short-Term Projections, 2022-2024, July 2023     96584
Colorado Short-Term Projections, 2023-2025, March 2024    91198
Colorado Short-Term Projections, 2016-2018, June 2016     74574
Name: perioddesc, dtype: int64
Colorado Short-Term Projections, 2016-2018, June 2016        372870
Colorado Short-Term Projections, 2020-2022, December 2019     86350
Name: perioddesc, dtype: int64


In [35]:
print(dfs4['shproj.csv']['periodid'].value_counts())
print(dfs3['shproj.csv']['periodid'].value_counts())
print(dfs['shproj.csv']['periodid'].value_counts())

69    74574
Name: periodid, dtype: int64
69    74574
Name: periodid, dtype: int64
69    372870
76     86350
Name: periodid, dtype: int64


In [13]:
dfs4['lngproj.csv'].compare(dfs3['lngproj.csv'])

areatype       areatyname           area                      areaname  \
          self other       self other     self    other                 self   
0          NaN   NaN        NaN   NaN  19740.0  17820.0  Denver - Aurora MSA   
1          NaN   NaN        NaN   NaN  19740.0  17820.0  Denver - Aurora MSA   
2          NaN   NaN        NaN   NaN  19740.0  17820.0  Denver - Aurora MSA   
3          NaN   NaN        NaN   NaN  19740.0  17820.0  Denver - Aurora MSA   
4          NaN   NaN        NaN   NaN  19740.0  17820.0  Denver - Aurora MSA   
...        ...   ...        ...   ...      ...      ...                  ...   
96792      NaN   NaN        NaN   NaN  80002.0  80003.0   Southwest Colorado   
96793      NaN   NaN        NaN   NaN  80002.0  80003.0   Southwest Colorado   
96794      NaN   NaN        NaN   NaN  80002.0  80003.0   Southwest Colorado   
96795      NaN   NaN        NaN   NaN  80002.0  80003.0   Southwest Colorado   
96796      NaN   NaN        NaN   NaN  80002.0  80003.0   Southwest Colorado   

                            matincode        ... pctprojind       pctprojocc  \
                      other      self other  ...       self other       self   
0      Colorado Springs MSA     670.0  67.0  ...       1.05  0.09       2.91   
1      Colorado Springs MSA     670.0  67.0  ...       1.76  0.02       4.42   
2      Colorado Springs MSA     670.0  67.0  ...       0.01  0.00       2.59   
3      Colorado Springs MSA     670.0  67.0  ...       0.97  0.00      12.91   
4      Colorado Springs MSA     670.0  67.0  ...       0.41  0.01       5.93   
...                     ...       ...   ...  ...        ...   ...        ...   
96792    Northwest Colorado       NaN   NaN  ...       1.35  3.43      79.67   
96793    Northwest Colorado       NaN   NaN  ...       0.82  0.66       2.51   
96794    Northwest Colorado       NaN   NaN  ...       0.20  2.41       2.14   
96795    Northwest Colorado       NaN   NaN  ...       0.39  0.46      18.34   
96796    Northwest Colorado       NaN   NaN  ...       0.18  0.37      76.00   

               nchg           pchg           growrate          
       other   self other     self     other     self   other  
0       1.25 -154.0   4.0 -10.4265   22.2222  -1.0951  2.0270  
1       1.68   70.0   0.0   3.2543    0.0000   0.3208  0.0000  
2       2.70    NaN   NaN      NaN       NaN      NaN     NaN  
3       0.49   -7.0   0.0  -0.5705    0.0000  -0.0572  0.0000  
4       0.98   35.0   1.0   7.2464  100.0000   0.7020  7.1773  
...      ...    ...   ...      ...       ...      ...     ...  
96792   7.64   22.0  63.0  17.8862   20.7237   1.6591  1.9012  
96793   4.71    NaN   NaN  15.7895   20.3390   1.4768  1.8687  
96794  86.87    3.0  44.0  16.6667   20.5607   1.5534  1.8874  
96795   1.45    6.0   9.0  16.6667   22.5000   1.5534  2.0501  
96796  58.82    3.0   7.0  18.7500   21.2121   1.7334  1.9423  

[96797 rows x 34 columns]

In [16]:
dfs4["Income.csv"].loc[(dfs4["Income.csv"]['periodyear'] == 2024) & (dfs4["Income.csv"]['periodtype'] != 0),'periodtype'].value_counts()

Series([], Name: periodtype, dtype: int64)

In [24]:
dfs4["Ces.csv"].columns

Index(['stateabbrv', 'statename', 'stfips', 'areatyname', 'areaname', 'area',
       'periodyear', 'periodtype', 'pertypdesc', 'period', 'seriescode',
       'seriesttls', 'seriesdesc', 'adjusted', 'benchmark', 'prelim', 'empces',
       'empprodwrk', 'empfemale', 'hours', 'earnings', 'hourearn',
       'supprecord', 'supphe', 'supppw', 'suppfem', 'hoursallwrkr',
       'earningsallwrkr', 'hourearnallwrkr', 'suppheallwrkr'],
      dtype='object')

In [25]:
dfs4["Ces.csv"].loc[(dfs4["Ces.csv"]['periodyear'] == 2024),'period'].value_counts()

2    344
1    344
0    303
Name: period, dtype: int64

In [27]:
dfs3["Ces.csv"].loc[(dfs3["Ces.csv"]['periodyear'] == 2023),'period'].value_counts()

11    344
7     344
3     344
10    344
6     344
2     344
9     344
5     344
1     344
12    344
8     344
4     344
0     303
Name: period, dtype: int64

In [21]:
dfs3["Income.csv"]['periodyear'].value_counts().sort_index()

1929      4
1930      4
1931      4
1932      4
1933      4
       ... 
2015    201
2016    197
2017    132
2020     65
2021     65
Name: periodyear, Length: 91, dtype: int64

In [29]:
dfs["Ces.csv"].loc[(dfs["Ces.csv"]['periodyear'] == 2023),'period'].value_counts()

7    344
5    344
3    344
1    344
6    344
4    344
2    344
0    303
Name: period, dtype: int64

In [12]:
dfLng = pd.read_csv("/home/joe/bic_etl/cdle/cdle_LaborAndEmployment/data_transformed/lngproj.csv")
dfSh = pd.read_csv("/home/joe/bic_etl/cdle/cdle_LaborAndEmployment/data_transformed/shproj.csv")

In [13]:
dfLng.head()

,stateabbrv,statename,stfips,areatype,areatyname,area,areaname,periodid,perioddesc,periodtype,matincodty,matincode,matintitle,matoccodty,matoccode,matocctitl,estemp,projemp,pctestind,pctestocc,pctprojind,pctprojocc,nchg,pchg,growrate,aopeng,aopenr,aopent,suppress
0,CO,Colorado,8,21,Metropolitan Statistical Area,19740,Denver - Aurora MSA,32,"Colorado Long-Term Projections, 2022-2032, Jul...",1,2,0,Total All Industries,2,0,"Total, All Occupations",1716301,1967762,100.0,100.00,100.0,100.0,251461,14.6513,1.3766,NaN,NaN,NaN,0
1,CO,Colorado,8,21,Metropolitan Statistical Area,19740,Denver - Aurora MSA,32,"Colorado Long-Term Projections, 2022-2032, Jul...",1,2,67,"Self Employed and Unpaid Family Workers, All Jobs",2,0,"Total, All Occupations",116133,125966,100.0,6.77,100.0,6.4,9833,8.4670,0.8161,NaN,NaN,NaN,0
2,CO,Colorado,8,21,Metropolitan Statistical Area,19740,Denver - Aurora MSA,32,"Colorado Long-Term Projections, 2022-2032, Jul...",1,2,670,"Self Employed and Unpaid Family Workers, All Jobs",2,0,"Total, All Occupations",116133,125966,100.0,6.77,100.0,6.4,9833,8.4670,0.8161,NaN,NaN,NaN,0
3,CO,Colorado,8,21,Metropolitan Statistical Area,19740,Denver - Aurora MSA,32,"Colorado Long-Term Projections, 2022-2032, Jul...",1,2,671,"Total Self Employed and Unpaid Family Workers,...",2,0,"Total, All Occupations",116133,125966,100.0,6.77,100.0,6.4,9833,8.4670,0.8161,NaN,NaN,NaN,0
4,CO,Colorado,8,21,Metropolitan Statistical Area,19740,Denver - Aurora MSA,32,"Colorado Long-Term Projections, 2022-2032, Jul...",1,2,6010,"Self Employed Workers, All Jobs",2,0,"Total, All Occupations",116133,125966,100.0,6.77,100.0,6.4,9833,8.4670,0.8161,NaN,NaN,NaN,0


In [14]:
dfLng.shape

(96797, 29)

In [15]:
dfLng['perioddesc'].value_counts()

Colorado Long-Term Projections, 2022-2032, July 2023    96797
Name: perioddesc, dtype: int64

In [16]:
dfSh['perioddesc'].value_counts()

Colorado Short-Term Projections, 2023-2025, March 2024    91198
Name: perioddesc, dtype: int64

In [2]:
dfLngC = pd.read_csv("https://data.colorado.gov/resource/gyeb-jc69.csv?$limit=10000000")
dfShC = pd.read_csv("https://data.colorado.gov/resource/u2t6-bfhr.csv?$limit=10000000")

In [3]:
dfShC['perioddesc'].value_counts()

Colorado Short-Term Projections, 2023-2025, March 2024    91198
Name: perioddesc, dtype: int64

In [4]:
dfLngC['perioddesc'].value_counts()

Colorado Long-Term Projections, 2022-2032, July 2023    96797
Name: perioddesc, dtype: int64

In [36]:
print(dfLngC.shape)
print(dfShC.shape)

(96797, 29)
(91198, 29)


## School Programs

In [2]:
dfSchoolC = pd.read_csv("https://data.colorado.gov/resource/jnj7-fw37.csv?$limit=10000000")

In [3]:
dfSchoolC.shape

(10304, 13)

In [47]:
dfSchoolC.head()

,statename,city,areaname,areadesc,instname1,progtitle,progdesc,instowndes,insttydesc,latitude,longitude,url,zip
0,Colorado,NaN,Douglas County,Douglas County is the eighth most populous of ...,University of Phoenix-Colorado Campus,Accounting and Business/Management.,NaN,Private for profit institution,Four-year Colleges and Universities,39.551643,-104.873421,http://,NaN
1,Colorado,NaN,Douglas County,Douglas County is the eighth most populous of ...,University of Phoenix-Colorado Campus,Accounting.,NaN,Private for profit institution,Four-year Colleges and Universities,39.551643,-104.873421,http://,NaN
2,Colorado,NaN,Douglas County,Douglas County is the eighth most populous of ...,University of Phoenix-Colorado Campus,Behavioral Sciences.,NaN,Private for profit institution,Four-year Colleges and Universities,39.551643,-104.873421,http://,NaN
3,Colorado,NaN,Douglas County,Douglas County is the eighth most populous of ...,University of Phoenix-Colorado Campus,"Business Administration and Management, General.",NaN,Private for profit institution,Four-year Colleges and Universities,39.551643,-104.873421,http://,NaN
4,Colorado,NaN,Douglas County,Douglas County is the eighth most populous of ...,University of Phoenix-Colorado Campus,"Computer and Information Sciences,",NaN,Private for profit institution,Four-year Colleges and Universities,39.551643,-104.873421,http://,NaN


In [4]:
dfSchool = pd.read_csv("/home/joe/bic_etl/cdle/cdle_education/data_transformed/school_programs.csv")


In [5]:
dfSchool.shape

(10304, 13)

In [6]:
dfSchool.columns

Index(['statename', 'city', 'areaname', 'areadesc', 'instname1', 'progtitle',
       'progdesc', 'instowndes', 'insttydesc', 'latitude', 'longitude', 'url',
       'zip'],
      dtype='object')

In [13]:
dfSchool['longitude'].nunique()

120

In [9]:
dfSchool['instname1'].nunique()

NameError: name 'dfSchoolC' is not defined

In [72]:
dct = dict(zip(dfSchool["instname1"],dfSchool["progtitle"]))
dctC = dict(zip(dfSchoolC["instname1"],dfSchoolC["progtitle"]))

In [73]:
for inst,prg in dct.items():
    prgC = dctC[inst]
    if prg != prgC:
        print(inst,prg,prgC)

Intellitec College-Colorado Springs Physical Fitness Technician. Physical Fitness Technician. (NEW)
Pikes Peak Community College Wildland/Forest Firefighting and Investigation. Wildland/Forest Firefighting and Investigation. (NEW)
Heritage College-Denver Physical Fitness Technician. Physical Fitness Technician. (NEW)
Intellitec College-Grand Junction Physical Fitness Technician. Physical Fitness Technician. (NEW)
Aims Community College Wildland/Forest Firefighting and Investigation. Wildland/Forest Firefighting and Investigation. (NEW)


In [75]:
for instC,prgC in dctC.items():
    prg = dct[instC]
    if prg != prgC:
        print(instC)
        print(prg)        
        print(prgC)   
        print("---------")

Intellitec College-Colorado Springs
Physical Fitness Technician.
Physical Fitness Technician. (NEW)
---------
Pikes Peak Community College
Wildland/Forest Firefighting and Investigation.
Wildland/Forest Firefighting and Investigation. (NEW)
---------
Heritage College-Denver
Physical Fitness Technician.
Physical Fitness Technician. (NEW)
---------
Intellitec College-Grand Junction
Physical Fitness Technician.
Physical Fitness Technician. (NEW)
---------
Aims Community College
Wildland/Forest Firefighting and Investigation.
Wildland/Forest Firefighting and Investigation. (NEW)
---------


In [ ]:
fin = open("/home/joe/bic_etl/cdle/cdle_education/data_transformed/school_programs.csv")
hist={}
for line in fin:
    spl = line.split(",")
    nn=len(spl)
    if nn in hist:
        hist[nn]+=1
    else:
        hist[nn]=1
        
    if nn != 13:
        print(nn)
        print(line)
        print("--------")

In [77]:
hist

{13: 1153,
 18: 597,
 19: 813,
 22: 416,
 23: 372,
 14: 1689,
 15: 814,
 16: 786,
 20: 613,
 17: 635,
 25: 152,
 24: 193,
 26: 239,
 31: 88,
 28: 363,
 21: 327,
 27: 550,
 29: 148,
 30: 142,
 32: 89,
 33: 74,
 37: 8,
 34: 22,
 35: 13,
 36: 4,
 38: 5}

In [79]:
dfSchool.shape

(10304, 13)

In [27]:
dfSchool.to_csv("school.csv",index=False)

In [29]:
dfSchoolNew = pd.read_csv("school.csv")

In [17]:
dfSchoolNew.shape

(10304, 13)

In [84]:
dfSchoolC['areadesc'].values

array(["Douglas County is the eighth most populous of the 64 counties of the state of Colorado, in the United States. The county is located midway between Colorado's two largest cities: Denver and Colorado Springs. The United States Census Bureau that the county population was 285,465 in 2010 census, a 62.4% increase since the 2000 census, making Douglas County one of the fastest growing counties in the United States. Douglas County is part of the Denver-Aurora Metropolitan Statistical Area and the Denver-Aurora-Boulder Combined Statistical Area. The county seat is Castle Rock. Douglas County has the highest median household income of any Colorado county or statistical equivalent. It is ranked eighth nationally in that category, and has the highest of any county or equivalent not in the northeastern US.",
       "Douglas County is the eighth most populous of the 64 counties of the state of Colorado, in the United States. The county is located midway between Colorado's two largest citie

In [7]:
histC={}
for col in dfSchoolC.columns:
    vals = dfSchoolC[col].values.tolist()
    for val in vals:
        if isinstance(val,str):
           if col not in histC:
                histC[col]={}
           for c in val:
               nn = ord(c)
               if nn in histC[col]:
                    histC[col][nn]+=1
               else:
                    histC[col][nn]=1
            

In [ ]:
for col,chrs in histC.items():
    for chr,cnt in sorted(chrs.items()):
        print(col,chr,cnt)
    print("------------------")

In [ ]:
vals = dfSchoolC['areadesc'].values.tolist()

for val in vals:
    hit=False
    if isinstance(val,str):
        for nn,chr in enumerate(val):
            if ord(chr) == 34:
                print(nn,chr,ord(chr))
                hit=True
        if hit:
            print(val)
        print("---")

In [19]:
dfSchool['progdesc'].value_counts()

A program that generally prepares individuals to plan, organize, direct, and control the functions and processes of a firm or organization.  Includes instruction in management theory, human resources management and behavior, accounting and other quantitat,Public Institution"                46
Any single instructional program in liberal arts and sciences, general studies and humanities not listed above.                                                                                                                                                                                    45
A program that generally prepares individuals to cut, trim, and style scalp, facial, and body hair; apply cosmetic preparations; perform manicures and pedicures; massage the head and extremities; and prepare for practice as licensed cosmetologists in spec,Private for profit institution"    44
A program that is a structured combination of the arts, biological and physical sciences, social sciences, and humanit

In [20]:
dfSchool.isna().sum()

statename        0
city            85
areaname         0
areadesc      1243
instname1        0
progtitle        0
progdesc      5542
instowndes       0
insttydesc       0
latitude         0
longitude        0
url              0
zip           3914
dtype: int64

In [21]:
dfSchool.shape

(10304, 13)

In [22]:
dfSchool.columns

Index(['statename', 'city', 'areaname', 'areadesc', 'instname1', 'progtitle',
       'progdesc', 'instowndes', 'insttydesc', 'latitude', 'longitude', 'url',
       'zip'],
      dtype='object')

In [ ]:
dfSchool['instname1'].value_counts()

In [ ]:
dfSchoolNew.loc[dfSchoolNew['instname1'] == "Western State Colorado University"]

In [30]:
dfSchool = pd.read_csv("/home/joe/bic_etl/cdle/cdle_education/data_transformed/school_programs.csv",encoding="latin")


In [31]:
dfSchool.loc[dfSchool['instname1'] == "Western State Colorado University"]

KeyError: 'instname1'

In [3]:
dfSchoolMan = pd.read_csv("/home/joe/bic_etl/cdle/cdle_education/data_transformed/school_programs_manual.csv")


In [4]:
dfSchoolMan.columns

Index(['statename', 'city', 'address', 'areaname', 'areadesc', 'instname1',
       'progtitle', 'progdesc', 'instowndes', 'insttydesc', 'latitude',
       'longitude', 'url', 'zip'],
      dtype='object')

In [ ]:
dfSchoolMan['zip'].value_counts()

In [8]:
dfSchoolMan.isna().sum()

statename        0
city            85
address         85
areaname         0
areadesc      1243
instname1        0
progtitle        0
progdesc      5542
instowndes       0
insttydesc       0
latitude         0
longitude        0
url              0
zip             85
dtype: int64

In [15]:
dfSchoolMan.groupby(["instname1","progtitle",'address']).count()

statename  \
instname1                               progtitle                                          address                        
Academy of Natural Therapy Inc          Massage Therapy/Therapeutic Massage.               625 8th Ave                2   
                                        Somatic Bodywork.                                  625 8th Ave                1   
Adams State University                  American Government and Politics (United States).  208 Edgemont Blvd          8   
                                        Biology/Biological Sciences, General.              208 Edgemont Blvd          8   
                                        Business Administration and Management, General.   208 Edgemont Blvd          8   
...                                                                                                                 ...   
Westwood College-Denver North           Web Page, Digital/Multimedia and Information Re... 7350 N Broadway            7   
Xenon International Academy-Denver      Aesthetician/Esthetician and Skin Care Specialist. 2231 S Peoria             10   
                                        Cosmetology/Cosmetologist, General.                2231 S Peoria             10   
                                        Nail Technician/Specialist and Manicurist.         2231 S Peoria              5   
Yeshiva Toras Chaim Talmudical Seminary Talmudic Studies.                                  1555 Stuart St             1   

                                                                                                              city  \
instname1                               progtitle                                          address                   
Academy of Natural Therapy Inc          Massage Therapy/Therapeutic Massage.               625 8th Ave           2   
                                        Somatic Bodywork.                                  625 8th Ave           1   
Adams State University                  American Government and Politics (United States).  208 Edgemont Blvd     8   
                                        Biology/Biological Sciences, General.              208 Edgemont Blvd     8   
                                        Business Administration and Management, General.   208 Edgemont Blvd     8   
...                                                                                                            ...   
Westwood College-Denver North           Web Page, Digital/Multimedia and Information Re... 7350 N Broadway       7   
Xenon International Academy-Denver      Aesthetician/Esthetician and Skin Care Specialist. 2231 S Peoria        10   
                                        Cosmetology/Cosmetologist, General.                2231 S Peoria        10   
                                        Nail Technician/Specialist and Manicurist.         2231 S Peoria         5   
Yeshiva Toras Chaim Talmudical Seminary Talmudic Studies.                                  1555 Stuart St        1   

                                                                                                              areaname  \
instname1                               progtitle                                          address                       
Academy of Natural Therapy Inc          Massage Therapy/Therapeutic Massage.               625 8th Ave               2   
                                        Somatic Bodywork.                                  625 8th Ave               1   
Adams State University                  American Government and Politics (United States).  208 Edgemont Blvd         8   
                                        Biology/Biological Sciences, General.              208 Edgemont Blvd         8   
                                        Business Administration and Management, General.   208 Edgemont Blvd         8   
...                                                                                                                ...   

In [23]:
dfSchoolMan.loc[(dfSchoolMan['instname1'] == "Adams State University")  &
                (dfSchoolMan['progtitle'] == "American Government and Politics (United States).")]

,statename,city,address,areaname,areadesc,instname1,progtitle,progdesc,instowndes,insttydesc,latitude,longitude,url,zip
85,Colorado,Alamosa,208 Edgemont Blvd,Alamosa County ...,Alamosa County is one of the 64 counties of th...,Adams State University,American Government and Politics (United States).,NaN,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0
86,Colorado,Alamosa,208 Edgemont Blvd,Alamosa County ...,Alamosa County is one of the 64 counties of th...,Adams State University,American Government and Politics (United States).,A program that focuses on the systematic study...,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0
143,Colorado,Alamosa,208 Edgemont Blvd,Co Rural Workforce Consortium,NaN,Adams State University,American Government and Politics (United States).,NaN,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0
144,Colorado,Alamosa,208 Edgemont Blvd,Co Rural Workforce Consortium,NaN,Adams State University,American Government and Politics (United States).,A program that focuses on the systematic study...,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0
201,Colorado,Alamosa,208 Edgemont Blvd,Edwards MCA ...,Eagle & Lake counties,Adams State University,American Government and Politics (United States).,NaN,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0
202,Colorado,Alamosa,208 Edgemont Blvd,Edwards MCA ...,Eagle & Lake counties,Adams State University,American Government and Politics (United States).,A program that focuses on the systematic study...,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0
259,Colorado,Alamosa,208 Edgemont Blvd,Planning Region 3 ...,,Adams State University,American Government and Politics (United States).,NaN,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0
260,Colorado,Alamosa,208 Edgemont Blvd,Planning Region 3 ...,,Adams State University,American Government and Politics (United States).,A program that focuses on the systematic study...,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0


In [ ]:
for inst in dfSchoolMan['instname1'].unique():
    dftmp = dfSchoolMan.loc[dfSchoolMan['instname1'] == inst]
#    dftmp.sort_values(by=['progtitle','areaname'],inplace=True)
    for 
    

### Fix School Data

In [5]:
dfbad = dfSchoolMan.loc[dfSchoolMan['progdesc'].isna()]
dfgood = dfSchoolMan.loc[~dfSchoolMan['progdesc'].isna()]

In [6]:
tot=0
histbad = {}
dftmp = pd.DataFrame(columns=dfgood.columns)
dfdel = pd.DataFrame(columns=dfgood.columns)

for index,row in dfbad.iterrows():
    inst = row['instname1']
    prog = row['progtitle']
    area = row['areaname']
    url = row['url']
    ownd = row['instowndes']
    indes = row['insttydesc']
    if inst not in histbad:
        histbad[inst]={}
        histbad[inst]['count']=0
        histbad[inst]['bad']=0
        
    histbad[inst]['count']+=1
    nn = dfgood.loc[(dfgood['instname1'] == inst) & 
                    (dfgood['progtitle'] == prog) & 
                    (dfgood['areaname']==area) & 
                    (dfgood['url'] == url) & 
                    (dfgood['instowndes'] == ownd) & 
                    (dfgood['insttydesc'] == indes)                   
                   ].shape[0]
    if nn == 0: # There are no other records, so we need to keep this record
        tot+=1
        histbad[inst]['bad']+=1
        dftmp = dftmp.append(row)
    elif nn == 1: # There IS another record and the descripiton is NOT missing, so we can delete this record with the missing description
        dfdel=dfdel.append(row)
    else:
        print(inst,prog,area)
        
        
print(dfbad.shape[0],tot)
  

5542 1873


In [7]:
dfdel.shape

(3669, 14)

In [ ]:
for inst,res in histbad.items():
    print(inst,res['count'],res['bad'])

In [49]:
dfdel.head()

,statename,city,address,areaname,areadesc,instname1,progtitle,progdesc,instowndes,insttydesc,latitude,longitude,url,zip
85,Colorado,Alamosa,208 Edgemont Blvd,Alamosa County ...,Alamosa County is one of the 64 counties of th...,Adams State University,American Government and Politics (United States).,NaN,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0
87,Colorado,Alamosa,208 Edgemont Blvd,Alamosa County ...,Alamosa County is one of the 64 counties of th...,Adams State University,"Biology/Biological Sciences, General.",NaN,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0
89,Colorado,Alamosa,208 Edgemont Blvd,Alamosa County ...,Alamosa County is one of the 64 counties of th...,Adams State University,"Business Administration and Management, General.",NaN,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0
91,Colorado,Alamosa,208 Edgemont Blvd,Alamosa County ...,Alamosa County is one of the 64 counties of th...,Adams State University,"Business Administration, Management and Operat...",NaN,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0
93,Colorado,Alamosa,208 Edgemont Blvd,Alamosa County ...,Alamosa County is one of the 64 counties of th...,Adams State University,"Business/Commerce, General.",NaN,Public Institution,Four-year Colleges and Universities,0.0,0.0,http://www.adams.edu,81101.0


In [35]:
4762+1873

6635

In [36]:
5542-1873

3669

In [8]:
## Add the records with missing descriptions that do not have a corresponding non-missing description record back for output 
dfgood = pd.concat([dfgood,dftmp])

In [13]:
for index,row in dfdel.iterrows():
    inst = row['instname1']
    prog = row['progtitle']
    area = row['areaname']
    url = row['url']
    ownd = row['instowndes']
    indes = row['insttydesc']
 
    histbad[inst]['count']+=1
    hld= dfgood.loc[(dfgood['instname1'] == inst) & 
                    (dfgood['progtitle'] == prog) & 
                    (dfgood['areaname']==area) & 
                    (dfgood['url'] == url) & 
                    (dfgood['instowndes'] == ownd) & 
                    (dfgood['insttydesc'] == indes)                   
                   ]
    nn=hld.shape[0]
    
    if nn == 1:
        mm = hld['progdesc'].isna().sum()
        if mm != 0:
            print("ROH ROH")
            print(hld.head())
            print("----")
    else:
        print("BAD BAD BAD",nn)
        print(hld.head())
        print("----")

In [14]:
longs = dfSchool['longitude'].values.tolist()

In [20]:
display(dfSchool.iloc[85])


statename                                              Colorado
city                                                    Alamosa
areaname      Alamosa County                                ...
areadesc      Alamosa County is one of the 64 counties of th...
instname1                                Adams State University
progtitle     American Government and Politics (United States).
progdesc                                                    NaN
instowndes                                   Public Institution
insttydesc                  Four-year Colleges and Universities
latitude                                                    0.0
longitude                                              0.000000
url                                        http://www.adams.edu
zip                                                     81101.0
Name: 85, dtype: object

In [18]:
dfSchool.iloc[86]

statename                                              Colorado
city                                                    Alamosa
areaname      Alamosa County                                ...
areadesc      Alamosa County is one of the 64 counties of th...
instname1                                Adams State University
progtitle     American Government and Politics (United States).
progdesc      A program that focuses on the systematic study...
instowndes                  Four-year Colleges and Universities
insttydesc                                             0.000000
latitude                                                    0.0
longitude                                  http://www.adams.edu
url                                                       81101
zip                                                         NaN
Name: 86, dtype: object

In [ ]:
for nn,lng in enumerate(longs):
    print(nn,lng)

In [22]:
file = "/home/joe/bic_etl/cdle/cdle_education/data_transformed/school_programs.csv"
fin = open(file)
lines = fin.readlines()


In [24]:
lines[86]

'"Colorado","Alamosa","Alamosa County                                              ","Alamosa County is one of the 64 counties of the state of Colorado of the United States. The county name is the Spanish language word for a ""grove of cottonwood trees."" The county population was 14,966 at U.S. Census 2000. The county seat is Alamosa.","Adams State University","American Government and Politics (United States).",NULL,"Public Institution","Four-year Colleges and Universities","0.000000","0.000000","http://www.adams.edu","81101"\n'

In [25]:
lines[87]

'"Colorado","Alamosa","Alamosa County                                              ","Alamosa County is one of the 64 counties of the state of Colorado of the United States. The county name is the Spanish language word for a ""grove of cottonwood trees."" The county population was 14,966 at U.S. Census 2000. The county seat is Alamosa.","Adams State University","American Government and Politics (United States).","A program that focuses on the systematic study of United States political institutions and behavior.  Includes instruction in American political theory, political parties and interest groups, state and local governments, Constitutional law, federalism and,"Public Institution","Four-year Colleges and Universities","0.000000","0.000000","http://www.adams.edu","81101"\n'

In [26]:
schoolNew = pd.read_csv("school_programs.csv")

In [27]:
schoolNew.loc[schoolNew['instname1'] == "Western State Colorado University"]


KeyError: 'instname1'

In [28]:
schoolNew.head()

,"""Colorado""",NULL,NULL.1,"""Douglas County ""","""Douglas County is the eighth most populous of the 64 counties of the state of Colorado, in the United States. The county is located midway between Colorado's two largest cities: Denver and Colorado Springs. The United States Census Bureau that the county population was 285,465 in 2010 census, a 62.4% increase since the 2000 census, making Douglas County one of the fastest growing counties in the United States. Douglas County is part of the Denver-Aurora Metropolitan Statistical Area and the Denver-Aurora-Boulder Combined Statistical Area. The county seat is Castle Rock. Douglas County has the highest median household income of any Colorado county or statistical equivalent. It is ranked eighth nationally in that category, and has the highest of any county or equivalent not in the northeastern US.""","""University of Phoenix-Colorado Campus""","""Accounting and Business/Management.""",NULL.2,"""Private for profit institution""","""Four-year Colleges and Universities""","""39.551643""","""-104.873421""","""http://""",NULL.3
0,"""Colorado""",NaN,NaN,"""Douglas County ...","""Douglas County is the eighth most populous of...","""University of Phoenix-Colorado Campus""","""Accounting.""",NaN,"""Private for profit institution""","""Four-year Colleges and Universities""","""39.551643""","""-104.873421""","""http://""",NaN
1,"""Colorado""",NaN,NaN,"""Douglas County ...","""Douglas County is the eighth most populous of...","""University of Phoenix-Colorado Campus""","""Behavioral Sciences.""",NaN,"""Private for profit institution""","""Four-year Colleges and Universities""","""39.551643""","""-104.873421""","""http://""",NaN
2,"""Colorado""",NaN,NaN,"""Douglas County ...","""Douglas County is the eighth most populous of...","""University of Phoenix-Colorado Campus""","""Business Administration and Management, Gener...",NaN,"""Private for profit institution""","""Four-year Colleges and Universities""","""39.551643""","""-104.873421""","""http://""",NaN
3,"""Colorado""",NaN,NaN,"""Douglas County ...","""Douglas County is the eighth most populous of...","""University of Phoenix-Colorado Campus""","""Computer and Information Sciences,""",NaN,"""Private for profit institution""","""Four-year Colleges and Universities""","""39.551643""","""-104.873421""","""http://""",NaN
4,"""Colorado""",NaN,NaN,"""Douglas County ...","""Douglas County is the eighth most populous of...","""University of Phoenix-Colorado Campus""","""Computer and Information Systems Security/Inf...",NaN,"""Private for profit institution""","""Four-year Colleges and Universities""","""39.551643""","""-104.873421""","""http://""",NaN


In [2]:
df = pd.read_csv("school_programs.csv")

In [3]:
df.head()

,statename,city,areaname,areadesc,instname1,progtitle,progdesc,instowndes,insttydesc,latitude,longitude,url,zip
0,Colorado,NaN,Douglas County ...,Douglas County is the eighth most populous of ...,University of Phoenix-Colorado Campus,Accounting and Business/Management.,NaN,Private for profit institution,Four-year Colleges and Universities,39.551643,-104.873421,http://,NaN
1,Colorado,NaN,Douglas County ...,Douglas County is the eighth most populous of ...,University of Phoenix-Colorado Campus,Accounting.,NaN,Private for profit institution,Four-year Colleges and Universities,39.551643,-104.873421,http://,NaN
2,Colorado,NaN,Douglas County ...,Douglas County is the eighth most populous of ...,University of Phoenix-Colorado Campus,Behavioral Sciences.,NaN,Private for profit institution,Four-year Colleges and Universities,39.551643,-104.873421,http://,NaN
3,Colorado,NaN,Douglas County ...,Douglas County is the eighth most populous of ...,University of Phoenix-Colorado Campus,"Business Administration and Management, General.",NaN,Private for profit institution,Four-year Colleges and Universities,39.551643,-104.873421,http://,NaN
4,Colorado,NaN,Douglas County ...,Douglas County is the eighth most populous of ...,University of Phoenix-Colorado Campus,"Computer and Information Sciences,",NaN,Private for profit institution,Four-year Colleges and Universities,39.551643,-104.873421,http://,NaN


In [4]:
df['longitude'].value_counts()

 0.000000      9714
-104.986386     196
-105.000000     154
-104.822841      66
-104.873421      60
-104.820000      30
-104.980000      28
-104.875949      20
-104.821001      13
-104.780941       7
-104.787967       5
-105.053321       4
-105.200000       3
-104.967732       1
-105.043282       1
-105.080647       1
-105.065051       1
Name: longitude, dtype: int64